# YOLOX-X — DIMER anchor-free detection and bounded detection fine-tuning (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/yolox-x-detection-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/yolox-x-detection-pipeline/blob/main/tutorials/yolox_x_detection_finetune_colab.ipynb) [![Upstream](https://img.shields.io/badge/Upstream-Megvii--BaseDetection%2FYOLOX-181717?style=flat&logo=github&logoColor=white)](https://github.com/Megvii-BaseDetection/YOLOX) [![Weights](https://img.shields.io/badge/Weights-yolox__x.pth%20%400.1.1rc0-blue?style=flat)](https://github.com/Megvii-BaseDetection/YOLOX/releases/tag/0.1.1rc0) [![arXiv](https://img.shields.io/badge/arXiv-2107.08430-b31b1b.svg)](https://arxiv.org/abs/2107.08430) [![License](https://img.shields.io/badge/License-Apache--2.0-green.svg)](https://github.com/Megvii-BaseDetection/YOLOX/blob/main/LICENSE)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** anchor-free object detection over the 80 COCO classes, and a bounded SimOTA fine-tune that re-heads the detector onto your own class vocabulary, evaluates it against a held-out split with COCO-style average precision, and exports a reloadable artifact

**This notebook is standalone.** It carries the repository's package (10 modules under `src/yolox_x_detection_pipeline/`, at revision `9ecb26c4d8ca`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the immutable GitHub release assets of upstream tag `0.1.1rc0` (~793 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, downloads and digest-verifies the pinned release asset, draws the COCO demonstration scene and scores it, builds and validates the labelled adaptation dataset, splits it, measures the pre-adaptation baseline on the held-out part, **runs the bounded SimOTA fine-tune**, re-scores the held-out split, detects on unseen images, exports the adapted artifact, reloads it from disk and confirms the reloaded model scores identically, and writes every result as JSON with provenance. Nothing is skipped behind a default-off flag, and the path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5, RUN7, FT2).

**Bring Your Own Data:** Two BYOD branches are provided and both are optional and off by default. `USE_BYOD_IMAGE` runs your own image through the same validation and detection contract as the sample. `USE_BYOD_DATASET` takes your own labelled records and runs them through the *same* local stages the sample used — validate, split, baseline, fine-tune, evaluate — rather than only detecting with them, because this is an adaptation profile (NOTEBOOK_SPEC 2.0 DAT14, §25.10). The expected record shape and the ceilings are stated in the Prerequisites and printed by the cell; uploads stay inside this runtime.

YOLOX-X is a small anchor-free detector: a CSPDarknet backbone and a PAFPN neck feed a **decoupled head** that emits, at each of 8400 anchor points over a 640×640 letterboxed image, one box, one objectness logit and one logit per class. There are no anchor boxes to tune and no class softmax — objectness and class probability are independent sigmoids whose product is the score, and per-class NMS does the rest. At training time the same head runs **SimOTA**: it decides for itself which anchor points should be responsible for which ground-truth object, by solving a small assignment problem over an IoU-and-classification cost, instead of using a fixed rule.

That last property is why this notebook can fine-tune a detector in a few minutes on a CPU. **The default path really adapts the model:** it re-heads YOLOX onto a three-class sign vocabulary that does not exist in COCO, measures a pre-adaptation baseline, runs a bounded SimOTA fine-tune with the backbone frozen, scores the result against a held-out split it never trained on, runs the adapted model on unseen images, exports the weights as one artifact and reloads that artifact as if from a cold start. Every number you will see comes from cells in this notebook, in this runtime.

Two facts about the input pipeline are load-bearing and the notebook will show you both. Channel order is **BGR**, raw 0–255, with no `/255` and no mean/std normalisation — that is what upstream feeds the network, and handing it RGB instead measurably degrades detection. And the letterbox pads with grey 114 rather than black. The carried package owns both, so you do not have to.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees and which of its modules are vendored upstream YOLOX rather than DIMER code; stage and digest-verify an immutable GitHub release asset (not a Hub revision) before anything unpickles it; run COCO detection on a drawn scene and read the objectness × class score, the two caller-owned thresholds and the per-object `box_iou` correctly, including one object the pretrained model gets wrong; build and validate a labelled detection dataset in the record shape the fine-tuner accepts; split it and measure a **baseline before adapting**; run a bounded SimOTA fine-tune onto a new class vocabulary; score it with COCO-style AP@[.50:.95] and AP50 on the held-out split; detect on images from an unseen seed; and export, reload and re-verify the resulting artifact.

**This notebook does not demonstrate:** real traffic-sign detection or any deployment claim — the adaptation dataset is drawn in code, so the fine-tuned model has learned these renderings and nothing about photographs; published COCO numbers (upstream reports 51.5 AP for YOLOX-X on COCO test-dev, which this notebook neither reproduces nor checks, and the average-precision helper here is a small faithful implementation without pycocotools' area ranges or crowd handling); upstream's mosaic/mixup augmentation, its 300-epoch schedule, its EMA and its learning-rate warmup, none of which a bounded tutorial run uses; the L1 box-refinement term, which upstream only enables for the last 15 epochs; full-network fine-tuning (the default freezes the backbone, and the repository records that unfreezing it at this learning rate collapses the model); instance segmentation, tracking, batched or video inference, quantisation, ONNX/TensorRT export, and upstream's latency figures.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). **CPU is enough and is the documented default** — this profile is float32 on both CPU and GPU, and CUDA is used automatically when present. For scale: on the repository's Windows CPU venv (Intel Core Ultra 9 275HX) loading takes 4.5 s, one `detect` 0.42 s, and the whole six-epoch fine-tune 130 s. This is the large YOLOX variant — roughly four times the per-image cost of YOLOX-S — so a hosted CPU runtime will be slower still; budget several minutes for the adaptation cell, or attach a GPU. The pinned `torch==2.14.0` install and the 793 MB checkpoint are the large downloads.
- **Knowledge:** basic Python and PIL; what a bounding box in xyxy pixel coordinates is; what intersection-over-union measures; roughly what average precision summarises. You do **not** need to know YOLOX internals — SimOTA assignment and the loss are upstream's code, carried and called, not reimplemented here.
- **Data:** everything is drawn in code by the carried `samples` module, so nothing is downloaded and no private data is needed: one 640×640 COCO demonstration scene with reference boxes, and a deterministic 40-image labelled sign dataset for the adaptation. Optional BYOD is gated off by default. Expected BYOD input: for detection, one image decodable by Pillow, sides 16–4096 px; for adaptation, a list of `{{'image': PIL.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': [name, ...]}}` records with boxes in that image's own pixels. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so; uploaded inputs stay in this runtime and are not sent to any inference API.
- **External access:** `github.com` only, to download the pinned `yolox_x.pth` release asset (~793 MB) from upstream tag `0.1.1rc0`. YOLOX publishes no Hugging Face model repository, so the checkpoint is a release asset rather than a Hub revision; the SHA-256 in the manifest below is what makes that download trustworthy. No credentials are required, and no code is fetched from any repository — the YOLOX model code you need is carried in Section 2.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `torchvision`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'yolox-x-detection-pipeline',
    'repository_revision': '9ecb26c4d8ca56999424088d6f012983b52a32ad',
    'embedded_module': 'src/yolox_x_detection_pipeline/pipeline.py',
    'embedded_modules': ['src/yolox_x_detection_pipeline/coco_classes.py', 'src/yolox_x_detection_pipeline/losses.py', 'src/yolox_x_detection_pipeline/network_blocks.py', 'src/yolox_x_detection_pipeline/ops.py', 'src/yolox_x_detection_pipeline/samples.py', 'src/yolox_x_detection_pipeline/darknet.py', 'src/yolox_x_detection_pipeline/pipeline.py', 'src/yolox_x_detection_pipeline/yolo_head.py', 'src/yolox_x_detection_pipeline/yolo_pafpn.py', 'src/yolox_x_detection_pipeline/yolox.py'],
    'module_sha256': '7705956dfffb5ed2ac2c09356c4bc273e543fbf37723f00754480598c6c0f7ff',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, torchvision, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'torchvision': torchvision.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/yolox_x_detection_pipeline/` @ `9ecb26c4d8ca`)

The next 10 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/10:** `src/yolox_x_detection_pipeline/coco_classes.py`

In [ ]:
#!/usr/bin/env python3
# -*- coding:utf-8 -*-
# Copyright (c) Megvii, Inc. and its affiliates.

COCO_CLASSES = (
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "airplane",
    "bus",
    "train",
    "truck",
    "boat",
    "traffic light",
    "fire hydrant",
    "stop sign",
    "parking meter",
    "bench",
    "bird",
    "cat",
    "dog",
    "horse",
    "sheep",
    "cow",
    "elephant",
    "bear",
    "zebra",
    "giraffe",
    "backpack",
    "umbrella",
    "handbag",
    "tie",
    "suitcase",
    "frisbee",
    "skis",
    "snowboard",
    "sports ball",
    "kite",
    "baseball bat",
    "baseball glove",
    "skateboard",
    "surfboard",
    "tennis racket",
    "bottle",
    "wine glass",
    "cup",
    "fork",
    "knife",
    "spoon",
    "bowl",
    "banana",
    "apple",
    "sandwich",
    "orange",
    "broccoli",
    "carrot",
    "hot dog",
    "pizza",
    "donut",
    "cake",
    "chair",
    "couch",
    "potted plant",
    "bed",
    "dining table",
    "toilet",
    "tv",
    "laptop",
    "mouse",
    "remote",
    "keyboard",
    "cell phone",
    "microwave",
    "oven",
    "toaster",
    "sink",
    "refrigerator",
    "book",
    "clock",
    "vase",
    "scissors",
    "teddy bear",
    "hair drier",
    "toothbrush",
)

**Module 2/10:** `src/yolox_x_detection_pipeline/losses.py` (carried verbatim; see the note above)

In [ ]:
#!/usr/bin/env python
# -*- encoding: utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

import torch
import torch.nn as nn


class IOUloss(nn.Module):
    def __init__(self, reduction="none", loss_type="iou"):
        super(IOUloss, self).__init__()
        self.reduction = reduction
        self.loss_type = loss_type

    def forward(self, pred, target):
        assert pred.shape[0] == target.shape[0]

        pred = pred.view(-1, 4)
        target = target.view(-1, 4)
        tl = torch.max(
            (pred[:, :2] - pred[:, 2:] / 2), (target[:, :2] - target[:, 2:] / 2)
        )
        br = torch.min(
            (pred[:, :2] + pred[:, 2:] / 2), (target[:, :2] + target[:, 2:] / 2)
        )

        area_p = torch.prod(pred[:, 2:], 1)
        area_g = torch.prod(target[:, 2:], 1)

        en = (tl < br).type(tl.type()).prod(dim=1)
        area_i = torch.prod(br - tl, 1) * en
        area_u = area_p + area_g - area_i
        iou = (area_i) / (area_u + 1e-16)

        if self.loss_type == "iou":
            loss = 1 - iou ** 2
        elif self.loss_type == "giou":
            c_tl = torch.min(
                (pred[:, :2] - pred[:, 2:] / 2), (target[:, :2] - target[:, 2:] / 2)
            )
            c_br = torch.max(
                (pred[:, :2] + pred[:, 2:] / 2), (target[:, :2] + target[:, 2:] / 2)
            )
            area_c = torch.prod(c_br - c_tl, 1)
            giou = iou - (area_c - area_u) / area_c.clamp(1e-16)
            loss = 1 - giou.clamp(min=-1.0, max=1.0)

        if self.reduction == "mean":
            loss = loss.mean()
        elif self.reduction == "sum":
            loss = loss.sum()

        return loss

**Module 3/10:** `src/yolox_x_detection_pipeline/network_blocks.py` (carried verbatim; see the note above)

In [ ]:
#!/usr/bin/env python
# -*- encoding: utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

import torch
import torch.nn as nn


class SiLU(nn.Module):
    """export-friendly version of nn.SiLU()"""

    @staticmethod
    def forward(x):
        return x * torch.sigmoid(x)


def get_activation(name="silu", inplace=True):
    if name == "silu":
        module = nn.SiLU(inplace=inplace)
    elif name == "relu":
        module = nn.ReLU(inplace=inplace)
    elif name == "lrelu":
        module = nn.LeakyReLU(0.1, inplace=inplace)
    else:
        raise AttributeError("Unsupported act type: {}".format(name))
    return module


class BaseConv(nn.Module):
    """A Conv2d -> Batchnorm -> silu/leaky relu block"""

    def __init__(
        self, in_channels, out_channels, ksize, stride, groups=1, bias=False, act="silu"
    ):
        super().__init__()
        # same padding
        pad = (ksize - 1) // 2
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=ksize,
            stride=stride,
            padding=pad,
            groups=groups,
            bias=bias,
        )
        self.bn = nn.BatchNorm2d(out_channels)
        self.act = get_activation(act, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

    def fuseforward(self, x):
        return self.act(self.conv(x))


class DWConv(nn.Module):
    """Depthwise Conv + Conv"""

    def __init__(self, in_channels, out_channels, ksize, stride=1, act="silu"):
        super().__init__()
        self.dconv = BaseConv(
            in_channels,
            in_channels,
            ksize=ksize,
            stride=stride,
            groups=in_channels,
            act=act,
        )
        self.pconv = BaseConv(
            in_channels, out_channels, ksize=1, stride=1, groups=1, act=act
        )

    def forward(self, x):
        x = self.dconv(x)
        return self.pconv(x)


class Bottleneck(nn.Module):
    # Standard bottleneck
    def __init__(
        self,
        in_channels,
        out_channels,
        shortcut=True,
        expansion=0.5,
        depthwise=False,
        act="silu",
    ):
        super().__init__()
        hidden_channels = int(out_channels * expansion)
        Conv = DWConv if depthwise else BaseConv
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=act)
        self.conv2 = Conv(hidden_channels, out_channels, 3, stride=1, act=act)
        self.use_add = shortcut and in_channels == out_channels

    def forward(self, x):
        y = self.conv2(self.conv1(x))
        if self.use_add:
            y = y + x
        return y


class ResLayer(nn.Module):
    "Residual layer with `in_channels` inputs."

    def __init__(self, in_channels: int):
        super().__init__()
        mid_channels = in_channels // 2
        self.layer1 = BaseConv(
            in_channels, mid_channels, ksize=1, stride=1, act="lrelu"
        )
        self.layer2 = BaseConv(
            mid_channels, in_channels, ksize=3, stride=1, act="lrelu"
        )

    def forward(self, x):
        out = self.layer2(self.layer1(x))
        return x + out


class SPPBottleneck(nn.Module):
    """Spatial pyramid pooling layer used in YOLOv3-SPP"""

    def __init__(
        self, in_channels, out_channels, kernel_sizes=(5, 9, 13), activation="silu"
    ):
        super().__init__()
        hidden_channels = in_channels // 2
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=activation)
        self.m = nn.ModuleList(
            [
                nn.MaxPool2d(kernel_size=ks, stride=1, padding=ks // 2)
                for ks in kernel_sizes
            ]
        )
        conv2_channels = hidden_channels * (len(kernel_sizes) + 1)
        self.conv2 = BaseConv(conv2_channels, out_channels, 1, stride=1, act=activation)

    def forward(self, x):
        x = self.conv1(x)
        x = torch.cat([x] + [m(x) for m in self.m], dim=1)
        x = self.conv2(x)
        return x


class CSPLayer(nn.Module):
    """C3 in yolov5, CSP Bottleneck with 3 convolutions"""

    def __init__(
        self,
        in_channels,
        out_channels,
        n=1,
        shortcut=True,
        expansion=0.5,
        depthwise=False,
        act="silu",
    ):
        """
        Args:
            in_channels (int): input channels.
            out_channels (int): output channels.
            n (int): number of Bottlenecks. Default value: 1.
        """
        # ch_in, ch_out, number, shortcut, groups, expansion
        super().__init__()
        hidden_channels = int(out_channels * expansion)  # hidden channels
        self.conv1 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=act)
        self.conv2 = BaseConv(in_channels, hidden_channels, 1, stride=1, act=act)
        self.conv3 = BaseConv(2 * hidden_channels, out_channels, 1, stride=1, act=act)
        module_list = [
            Bottleneck(
                hidden_channels, hidden_channels, shortcut, 1.0, depthwise, act=act
            )
            for _ in range(n)
        ]
        self.m = nn.Sequential(*module_list)

    def forward(self, x):
        x_1 = self.conv1(x)
        x_2 = self.conv2(x)
        x_1 = self.m(x_1)
        x = torch.cat((x_1, x_2), dim=1)
        return self.conv3(x)


class Focus(nn.Module):
    """Focus width and height information into channel space."""

    def __init__(self, in_channels, out_channels, ksize=1, stride=1, act="silu"):
        super().__init__()
        self.conv = BaseConv(in_channels * 4, out_channels, ksize, stride, act=act)

    def forward(self, x):
        # shape of x (b,c,w,h) -> y(b,4c,w/2,h/2)
        patch_top_left = x[..., ::2, ::2]
        patch_top_right = x[..., ::2, 1::2]
        patch_bot_left = x[..., 1::2, ::2]
        patch_bot_right = x[..., 1::2, 1::2]
        x = torch.cat(
            (
                patch_top_left,
                patch_bot_left,
                patch_top_right,
                patch_bot_right,
            ),
            dim=1,
        )
        return self.conv(x)

**Module 4/10:** `src/yolox_x_detection_pipeline/ops.py` (carried verbatim; see the note above)

In [ ]:
"""Box helpers and the NMS post-processor, vendored from upstream ``yolox/utils``.

Concatenated from ``yolox/utils/compat.py`` and ``yolox/utils/boxes.py`` at the revision recorded in
``docs/UPSTREAM.md``, minus their shebang lines. The upstream ``yolox/utils/__init__.py`` re-exports
modules that import ``cv2``, ``loguru``, ``thop`` and ``tabulate``; only the helpers below are on the
inference or training path, so vendoring them keeps this package's dependency set at torch,
torchvision, numpy and Pillow.
"""

# -*- coding:utf-8 -*-

import torch

_TORCH_VER = [int(x) for x in torch.__version__.split(".")[:2]]

__all__ = ["meshgrid"]


def meshgrid(*tensors):
    if _TORCH_VER >= [1, 10]:
        return torch.meshgrid(*tensors, indexing="ij")
    else:
        return torch.meshgrid(*tensors)
# -*- coding:utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

import numpy as np

import torch
import torchvision

__all__ = [
    "filter_box",
    "postprocess",
    "bboxes_iou",
    "matrix_iou",
    "adjust_box_anns",
    "xyxy2xywh",
    "xyxy2cxcywh",
]


def filter_box(output, scale_range):
    """
    output: (N, 5+class) shape
    """
    min_scale, max_scale = scale_range
    w = output[:, 2] - output[:, 0]
    h = output[:, 3] - output[:, 1]
    keep = (w * h > min_scale * min_scale) & (w * h < max_scale * max_scale)
    return output[keep]


def postprocess(prediction, num_classes, conf_thre=0.7, nms_thre=0.45, class_agnostic=False):
    box_corner = prediction.new(prediction.shape)
    box_corner[:, :, 0] = prediction[:, :, 0] - prediction[:, :, 2] / 2
    box_corner[:, :, 1] = prediction[:, :, 1] - prediction[:, :, 3] / 2
    box_corner[:, :, 2] = prediction[:, :, 0] + prediction[:, :, 2] / 2
    box_corner[:, :, 3] = prediction[:, :, 1] + prediction[:, :, 3] / 2
    prediction[:, :, :4] = box_corner[:, :, :4]

    output = [None for _ in range(len(prediction))]
    for i, image_pred in enumerate(prediction):

        # If none are remaining => process next image
        if not image_pred.size(0):
            continue
        # Get score and class with highest confidence
        class_conf, class_pred = torch.max(image_pred[:, 5: 5 + num_classes], 1, keepdim=True)

        conf_mask = (image_pred[:, 4] * class_conf.squeeze() >= conf_thre).squeeze()
        # Detections ordered as (x1, y1, x2, y2, obj_conf, class_conf, class_pred)
        detections = torch.cat((image_pred[:, :5], class_conf, class_pred.float()), 1)
        detections = detections[conf_mask]
        if not detections.size(0):
            continue

        if class_agnostic:
            nms_out_index = torchvision.ops.nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                nms_thre,
            )
        else:
            nms_out_index = torchvision.ops.batched_nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                detections[:, 6],
                nms_thre,
            )

        detections = detections[nms_out_index]
        if output[i] is None:
            output[i] = detections
        else:
            output[i] = torch.cat((output[i], detections))

    return output


def bboxes_iou(bboxes_a, bboxes_b, xyxy=True):
    if bboxes_a.shape[1] != 4 or bboxes_b.shape[1] != 4:
        raise IndexError

    if xyxy:
        tl = torch.max(bboxes_a[:, None, :2], bboxes_b[:, :2])
        br = torch.min(bboxes_a[:, None, 2:], bboxes_b[:, 2:])
        area_a = torch.prod(bboxes_a[:, 2:] - bboxes_a[:, :2], 1)
        area_b = torch.prod(bboxes_b[:, 2:] - bboxes_b[:, :2], 1)
    else:
        tl = torch.max(
            (bboxes_a[:, None, :2] - bboxes_a[:, None, 2:] / 2),
            (bboxes_b[:, :2] - bboxes_b[:, 2:] / 2),
        )
        br = torch.min(
            (bboxes_a[:, None, :2] + bboxes_a[:, None, 2:] / 2),
            (bboxes_b[:, :2] + bboxes_b[:, 2:] / 2),
        )

        area_a = torch.prod(bboxes_a[:, 2:], 1)
        area_b = torch.prod(bboxes_b[:, 2:], 1)
    en = (tl < br).type(tl.type()).prod(dim=2)
    area_i = torch.prod(br - tl, 2) * en  # * ((tl < br).all())
    return area_i / (area_a[:, None] + area_b - area_i)


def matrix_iou(a, b):
    """
    return iou of a and b, numpy version for data augenmentation
    """
    lt = np.maximum(a[:, np.newaxis, :2], b[:, :2])
    rb = np.minimum(a[:, np.newaxis, 2:], b[:, 2:])

    area_i = np.prod(rb - lt, axis=2) * (lt < rb).all(axis=2)
    area_a = np.prod(a[:, 2:] - a[:, :2], axis=1)
    area_b = np.prod(b[:, 2:] - b[:, :2], axis=1)
    return area_i / (area_a[:, np.newaxis] + area_b - area_i + 1e-12)


def adjust_box_anns(bbox, scale_ratio, padw, padh, w_max, h_max):
    bbox[:, 0::2] = np.clip(bbox[:, 0::2] * scale_ratio + padw, 0, w_max)
    bbox[:, 1::2] = np.clip(bbox[:, 1::2] * scale_ratio + padh, 0, h_max)
    return bbox


def xyxy2xywh(bboxes):
    bboxes[:, 2] = bboxes[:, 2] - bboxes[:, 0]
    bboxes[:, 3] = bboxes[:, 3] - bboxes[:, 1]
    return bboxes


def xyxy2cxcywh(bboxes):
    bboxes[:, 2] = bboxes[:, 2] - bboxes[:, 0]
    bboxes[:, 3] = bboxes[:, 3] - bboxes[:, 1]
    bboxes[:, 0] = bboxes[:, 0] + bboxes[:, 2] * 0.5
    bboxes[:, 1] = bboxes[:, 1] + bboxes[:, 3] * 0.5
    return bboxes

**Module 5/10:** `src/yolox_x_detection_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Deterministic in-code sample data: the COCO demonstration scene and the adaptation dataset.

Nothing here is downloaded and nothing needs torch — Pillow and numpy only — so the tutorial's
default path has no dataset dependency and the same generators are exercised by the repository's unit
tests and by the smoke run whose numbers the model card quotes.

Everything is drawn at ``INPUT_SIZE`` (640x640), which is also what keeps the letterbox a no-op on the
default path (see ``pipeline.preprocess``).

Two separate label vocabularies live here and must not be conflated:

* ``tutorial_scene`` returns references drawn from the checkpoint's own 80 **COCO** classes; it
  demonstrates the pretrained model and is not training data.
* ``sign_dataset`` returns records labelled with ``SIGN_CLASSES``, a three-class vocabulary that does
  **not** exist in COCO. It is the adaptation dataset, and a model fine-tuned on it answers in those
  three names only.
"""

from __future__ import annotations

import math
from typing import Any

import numpy as np
from PIL import Image, ImageDraw, ImageFont

SCENE_SIZE = (640, 640)

# The adaptation vocabulary. Deliberately not COCO names: "stop sign" exists in COCO, "yield-sign" and
# "speed-limit-sign" do not, and the hyphenated spellings keep the two vocabularies visually distinct
# in output. A model fine-tuned on this dataset predicts only these three.
SIGN_CLASSES: tuple[str, ...] = ("stop-sign", "yield-sign", "speed-limit-sign")


def _font(size: int) -> Any:
    return ImageFont.load_default(size=size)


def _street_background(draw: ImageDraw.ImageDraw, width: int, height: int) -> None:
    draw.rectangle([0, 0, width, height], fill=(232, 236, 240))
    draw.rectangle([0, int(height * 0.73), width, height], fill=(120, 124, 128))
    draw.rectangle([0, int(height * 0.73), width, int(height * 0.73) + 8], fill=(240, 240, 240))


def _octagon(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float, font: Any) -> list[float]:
    points = [
        (cx + r * math.cos(math.pi / 8 + i * math.pi / 4), cy + r * math.sin(math.pi / 8 + i * math.pi / 4))
        for i in range(8)
    ]
    draw.polygon(points, fill=(196, 30, 34), outline=(255, 255, 255))
    text = "STOP"
    draw.text(
        (cx - draw.textlength(text, font=font) / 2, cy - r * 0.27), text, fill=(255, 255, 255), font=font
    )
    # A regular octagon inscribed in a circle of radius r reaches only r*cos(pi/8) ~ 0.924r along
    # either axis, never r. Returning the circumscribing square would make the reference box 17%
    # larger in area than the shape it names, which caps the achievable IoU and deflates every AP
    # number computed against it. The extent is derived from the same vertices that were drawn.
    half = r * math.cos(math.pi / 8)
    return [cx - half, cy - half, cx + half, cy + half]


def _traffic_light(draw: ImageDraw.ImageDraw, box: tuple[float, float, float, float]) -> list[float]:
    x0, y0, x1, y1 = box
    draw.rounded_rectangle([x0, y0, x1, y1], radius=10, fill=(40, 42, 46))
    lamp_h = (y1 - y0) / 3
    for i, colour in enumerate([(220, 50, 40), (232, 190, 60), (70, 190, 90)]):
        top = y0 + 10 + i * lamp_h
        draw.ellipse([x0 + 8, top, x1 - 8, top + lamp_h - 12], fill=colour)
    draw.rectangle([(x0 + x1) / 2 - 6, y1, (x0 + x1) / 2 + 6, 480], fill=(110, 110, 114))
    return [x0, y0, x1, y1]


def _analogue_clock(
    draw: ImageDraw.ImageDraw, box: tuple[float, float, float, float], font: Any
) -> list[float]:
    """A clock face with tick marks and hour numerals.

    The numerals matter: a plain white disc with two hands is not recognised as a clock by this
    checkpoint, and worse, it makes the drawn ball elsewhere in the scene *become* the clock. The
    repository's release verification records that probe.
    """
    x0, y0, x1, y1 = box
    cx, cy, r = (x0 + x1) / 2, (y0 + y1) / 2, (x1 - x0) / 2
    draw.ellipse([x0, y0, x1, y1], fill=(252, 252, 250), outline=(24, 24, 24), width=7)
    for i in range(12):
        angle = -math.pi / 2 + i * math.pi / 6
        draw.line(
            [
                cx + (r - 14) * math.cos(angle),
                cy + (r - 14) * math.sin(angle),
                cx + (r - 6) * math.cos(angle),
                cy + (r - 6) * math.sin(angle),
            ],
            fill=(30, 30, 30),
            width=3,
        )
    for hour, step in ((12, 9), (3, 0), (6, 3), (9, 6)):
        angle = -math.pi / 2 + step * math.pi / 6
        tx, ty = cx + (r - 30) * math.cos(angle), cy + (r - 30) * math.sin(angle)
        draw.text((tx - 9, ty - 11), str(hour), fill=(24, 24, 24), font=font)
    draw.line(
        [cx, cy, cx + (r - 34) * math.cos(-math.pi / 3), cy + (r - 34) * math.sin(-math.pi / 3)],
        fill=(20, 20, 20),
        width=7,
    )
    draw.line(
        [cx, cy, cx + (r - 18) * math.cos(math.pi / 8), cy + (r - 18) * math.sin(math.pi / 8)],
        fill=(20, 20, 20),
        width=4,
    )
    draw.ellipse([cx - 5, cy - 5, cx + 5, cy + 5], fill=(20, 20, 20))
    return [x0, y0, x1, y1]


def _football(draw: ImageDraw.ImageDraw, box: tuple[float, float, float, float]) -> list[float]:
    x0, y0, x1, y1 = box
    cx, cy, r = (x0 + x1) / 2, (y0 + y1) / 2, (x1 - x0) / 2
    draw.ellipse([x0, y0, x1, y1], fill=(250, 250, 250), outline=(30, 30, 30), width=3)
    pentagon = [
        (
            cx + 0.36 * r * math.cos(-math.pi / 2 + i * 2 * math.pi / 5),
            cy + 0.36 * r * math.sin(-math.pi / 2 + i * 2 * math.pi / 5),
        )
        for i in range(5)
    ]
    draw.polygon(pentagon, fill=(28, 28, 28))
    for px, py in pentagon:
        draw.line([px, py, cx + (px - cx) * 2.6, cy + (py - cy) * 2.6], fill=(28, 28, 28), width=3)
    return [x0, y0, x1, y1]


def _bench(draw: ImageDraw.ImageDraw, box: tuple[float, float, float, float]) -> list[float]:
    x0, y0, x1, y1 = box
    draw.rectangle([x0, y0, x1, y0 + 16], fill=(150, 105, 60))
    draw.rectangle([x0, y0 + 26, x1, y0 + 40], fill=(150, 105, 60))
    draw.rectangle([x0 + 8, y0 + 16, x0 + 18, y1], fill=(120, 84, 48))
    draw.rectangle([x1 - 18, y0 + 16, x1 - 8, y1], fill=(120, 84, 48))
    return [x0, y0, x1, y1]


def tutorial_scene() -> tuple[Image.Image, dict[str, list[list[float]]]]:
    """The COCO demonstration scene: five drawn objects and their reference boxes, keyed by COCO label.

    These are drawn references on a rendered picture, not a labelled photographic dataset: they are
    enough for a per-object ``box_iou`` sanity check and nothing more.
    """
    image = Image.new("RGB", SCENE_SIZE, (232, 236, 240))
    draw = ImageDraw.Draw(image)
    _street_background(draw, *SCENE_SIZE)
    references: dict[str, list[list[float]]] = {}
    draw.rectangle([114, 270, 126, 480], fill=(110, 110, 114))
    references["stop sign"] = [_octagon(draw, 120, 200, 70, _font(22))]
    references["traffic light"] = [_traffic_light(draw, (300, 120, 356, 250))]
    references["clock"] = [_analogue_clock(draw, (460, 130, 600, 270), _font(18))]
    references["sports ball"] = [_football(draw, (240, 520, 320, 600))]
    references["bench"] = [_bench(draw, (440, 500, 610, 580))]
    return image, references


def blank_scene() -> Image.Image:
    """A featureless white image: the degenerate input every detector should be asked about."""
    return Image.new("RGB", SCENE_SIZE, (255, 255, 255))


def noise_scene(seed: int = 0) -> Image.Image:
    """Uniform RGB noise: structure-free input, for the same reason as ``blank_scene``."""
    rng = np.random.default_rng(seed)
    return Image.fromarray(rng.integers(0, 256, (*SCENE_SIZE[::-1], 3), dtype=np.uint8))


def _yield_sign(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float) -> list[float]:
    points = [(cx, cy + r), (cx - r * 0.95, cy - r * 0.75), (cx + r * 0.95, cy - r * 0.75)]
    draw.polygon(points, fill=(255, 255, 255), outline=(198, 32, 36))
    inner = [(cx, cy + r * 0.62), (cx - r * 0.62, cy - r * 0.5), (cx + r * 0.62, cy - r * 0.5)]
    draw.line([*inner, inner[0]], fill=(198, 32, 36), width=int(max(4, r * 0.22)))
    return [cx - r * 0.95, cy - r * 0.75, cx + r * 0.95, cy + r]


def _speed_limit_sign(draw: ImageDraw.ImageDraw, cx: float, cy: float, r: float, limit: int) -> list[float]:
    draw.ellipse(
        [cx - r, cy - r, cx + r, cy + r],
        fill=(255, 255, 255),
        outline=(198, 32, 36),
        width=int(max(4, r * 0.2)),
    )
    font = _font(int(max(12, r * 0.9)))
    text = str(limit)
    draw.text((cx - draw.textlength(text, font=font) / 2, cy - r * 0.55), text, fill=(30, 30, 30), font=font)
    return [cx - r, cy - r, cx + r, cy + r]


def sign_dataset(
    n_images: int = 40,
    *,
    seed: int = 0,
    max_objects: int = 3,
) -> list[dict[str, Any]]:
    """A deterministic labelled dataset over ``SIGN_CLASSES`` for the bounded fine-tune.

    Each record is ``{"image": PIL.Image, "boxes": [[x0, y0, x1, y1], ...], "labels": [name, ...]}`` —
    the record shape ``validate_dataset`` and ``finetune`` accept, and the shape a BYOD caller must
    produce from their own labelled images. Signs are placed on a non-overlapping grid so the boxes
    are exact by construction rather than approximate.

    It is synthetic drawn data, so a model fine-tuned on it learns to find *these renderings*. That is
    the point of a bounded tutorial adaptation and the reason its metrics are not a claim about real
    traffic signs.
    """
    if not 1 <= n_images <= 500:
        raise ValueError(f"n_images must be in 1..500, got {n_images}")
    if not 1 <= max_objects <= 6:
        raise ValueError(f"max_objects must be in 1..6, got {max_objects}")
    rng = np.random.default_rng(seed)
    width, height = SCENE_SIZE
    slots = [(x, y) for y in (150, 400) for x in (140, 360, 560)]
    records: list[dict[str, Any]] = []
    for _ in range(n_images):
        image = Image.new("RGB", SCENE_SIZE, (232, 236, 240))
        draw = ImageDraw.Draw(image)
        # A varied but never-black background, so the network cannot key on a constant canvas.
        tint = rng.integers(200, 245, 3)
        draw.rectangle([0, 0, width, height], fill=tuple(int(v) for v in tint))
        draw.rectangle([0, int(height * 0.72), width, height], fill=(118, 122, 126))
        n_objects = int(rng.integers(1, max_objects + 1))
        chosen = rng.permutation(len(slots))[:n_objects]
        boxes: list[list[float]] = []
        labels: list[str] = []
        for slot_index in chosen:
            cx, cy = slots[int(slot_index)]
            cx += float(rng.integers(-28, 29))
            cy += float(rng.integers(-28, 29))
            radius = float(rng.integers(46, 71))
            # Keep the whole sign inside the canvas: a clipped shape would make its reference box
            # loose, and a loose reference box turns every IoU and AP number below into noise.
            cx = min(max(cx, radius + 6), width - radius - 6)
            cy = min(max(cy, radius + 6), height - radius - 100)
            kind = SIGN_CLASSES[int(rng.integers(0, len(SIGN_CLASSES)))]
            draw.rectangle(
                [cx - 5, cy + radius * 0.6, cx + 5, min(height - 1, cy + radius + 90)], fill=(112, 112, 116)
            )
            if kind == "stop-sign":
                box = _octagon(draw, cx, cy, radius, _font(int(max(11, radius * 0.34))))
            elif kind == "yield-sign":
                box = _yield_sign(draw, cx, cy, radius)
            else:
                box = _speed_limit_sign(draw, cx, cy, radius, int(rng.choice([30, 50, 60, 80])))
            boxes.append(
                [
                    float(max(0.0, box[0])),
                    float(max(0.0, box[1])),
                    float(min(width, box[2])),
                    float(min(height, box[3])),
                ]
            )
            labels.append(kind)
        records.append({"image": image, "boxes": boxes, "labels": labels})
    return records

**Module 6/10:** `src/yolox_x_detection_pipeline/darknet.py` (carried verbatim; see the note above)

In [ ]:
#!/usr/bin/env python
# -*- encoding: utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

from torch import nn

# standalone rewrite (build_notebook.py): `from .network_blocks import BaseConv, CSPLayer, DWConv, Focus, ResLayer, SPPBottleneck` removed — names are kernel globals defined by the carried modules


class Darknet(nn.Module):
    # number of blocks from dark2 to dark5.
    depth2blocks = {21: [1, 2, 2, 1], 53: [2, 8, 8, 4]}

    def __init__(
        self,
        depth,
        in_channels=3,
        stem_out_channels=32,
        out_features=("dark3", "dark4", "dark5"),
    ):
        """
        Args:
            depth (int): depth of darknet used in model, usually use [21, 53] for this param.
            in_channels (int): number of input channels, for example, use 3 for RGB image.
            stem_out_channels (int): number of output channels of darknet stem.
                It decides channels of darknet layer2 to layer5.
            out_features (Tuple[str]): desired output layer name.
        """
        super().__init__()
        assert out_features, "please provide output features of Darknet"
        self.out_features = out_features
        self.stem = nn.Sequential(
            BaseConv(in_channels, stem_out_channels, ksize=3, stride=1, act="lrelu"),
            *self.make_group_layer(stem_out_channels, num_blocks=1, stride=2),
        )
        in_channels = stem_out_channels * 2  # 64

        num_blocks = Darknet.depth2blocks[depth]
        # create darknet with `stem_out_channels` and `num_blocks` layers.
        # to make model structure more clear, we don't use `for` statement in python.
        self.dark2 = nn.Sequential(
            *self.make_group_layer(in_channels, num_blocks[0], stride=2)
        )
        in_channels *= 2  # 128
        self.dark3 = nn.Sequential(
            *self.make_group_layer(in_channels, num_blocks[1], stride=2)
        )
        in_channels *= 2  # 256
        self.dark4 = nn.Sequential(
            *self.make_group_layer(in_channels, num_blocks[2], stride=2)
        )
        in_channels *= 2  # 512

        self.dark5 = nn.Sequential(
            *self.make_group_layer(in_channels, num_blocks[3], stride=2),
            *self.make_spp_block([in_channels, in_channels * 2], in_channels * 2),
        )

    def make_group_layer(self, in_channels: int, num_blocks: int, stride: int = 1):
        "starts with conv layer then has `num_blocks` `ResLayer`"
        return [
            BaseConv(in_channels, in_channels * 2, ksize=3, stride=stride, act="lrelu"),
            *[(ResLayer(in_channels * 2)) for _ in range(num_blocks)],
        ]

    def make_spp_block(self, filters_list, in_filters):
        m = nn.Sequential(
            *[
                BaseConv(in_filters, filters_list[0], 1, stride=1, act="lrelu"),
                BaseConv(filters_list[0], filters_list[1], 3, stride=1, act="lrelu"),
                SPPBottleneck(
                    in_channels=filters_list[1],
                    out_channels=filters_list[0],
                    activation="lrelu",
                ),
                BaseConv(filters_list[0], filters_list[1], 3, stride=1, act="lrelu"),
                BaseConv(filters_list[1], filters_list[0], 1, stride=1, act="lrelu"),
            ]
        )
        return m

    def forward(self, x):
        outputs = {}
        x = self.stem(x)
        outputs["stem"] = x
        x = self.dark2(x)
        outputs["dark2"] = x
        x = self.dark3(x)
        outputs["dark3"] = x
        x = self.dark4(x)
        outputs["dark4"] = x
        x = self.dark5(x)
        outputs["dark5"] = x
        return {k: v for k, v in outputs.items() if k in self.out_features}


class CSPDarknet(nn.Module):
    def __init__(
        self,
        dep_mul,
        wid_mul,
        out_features=("dark3", "dark4", "dark5"),
        depthwise=False,
        act="silu",
    ):
        super().__init__()
        assert out_features, "please provide output features of Darknet"
        self.out_features = out_features
        Conv = DWConv if depthwise else BaseConv

        base_channels = int(wid_mul * 64)  # 64
        base_depth = max(round(dep_mul * 3), 1)  # 3

        # stem
        self.stem = Focus(3, base_channels, ksize=3, act=act)

        # dark2
        self.dark2 = nn.Sequential(
            Conv(base_channels, base_channels * 2, 3, 2, act=act),
            CSPLayer(
                base_channels * 2,
                base_channels * 2,
                n=base_depth,
                depthwise=depthwise,
                act=act,
            ),
        )

        # dark3
        self.dark3 = nn.Sequential(
            Conv(base_channels * 2, base_channels * 4, 3, 2, act=act),
            CSPLayer(
                base_channels * 4,
                base_channels * 4,
                n=base_depth * 3,
                depthwise=depthwise,
                act=act,
            ),
        )

        # dark4
        self.dark4 = nn.Sequential(
            Conv(base_channels * 4, base_channels * 8, 3, 2, act=act),
            CSPLayer(
                base_channels * 8,
                base_channels * 8,
                n=base_depth * 3,
                depthwise=depthwise,
                act=act,
            ),
        )

        # dark5
        self.dark5 = nn.Sequential(
            Conv(base_channels * 8, base_channels * 16, 3, 2, act=act),
            SPPBottleneck(base_channels * 16, base_channels * 16, activation=act),
            CSPLayer(
                base_channels * 16,
                base_channels * 16,
                n=base_depth,
                shortcut=False,
                depthwise=depthwise,
                act=act,
            ),
        )

    def forward(self, x):
        outputs = {}
        x = self.stem(x)
        outputs["stem"] = x
        x = self.dark2(x)
        outputs["dark2"] = x
        x = self.dark3(x)
        outputs["dark3"] = x
        x = self.dark4(x)
        outputs["dark4"] = x
        x = self.dark5(x)
        outputs["dark5"] = x
        return {k: v for k, v in outputs.items() if k in self.out_features}

**Module 7/10:** `src/yolox_x_detection_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Anchor-free COCO object detection and bounded detection fine-tuning on the pinned YOLOX-X checkpoint.

The checkpoint is ``yolox_x.pth`` from the upstream GitHub release ``0.1.1rc0`` — YOLOX publishes no
Hugging Face model repository, so provenance here is a release asset URL plus the SHA-256 recorded in
``weights/yolox-x/dimer-base-manifest.json``, not a Hub revision. The model code is vendored from the
upstream repository at a pinned commit (``docs/UPSTREAM.md``); nothing is imported from the ``yolox``
distribution and no remote code is executed.

Two capabilities:

* ``detect`` — the 80 COCO classes, score-ordered xyxy pixel boxes, caller-owned confidence and NMS
  thresholds.
* ``finetune`` — a bounded SimOTA fine-tune onto a caller-supplied class vocabulary, re-heading the
  classification branch and keeping the pretrained backbone, neck, box and objectness weights. The
  result is exportable as a single artifact and reloadable from a fresh process.

Two things about the input pipeline are load-bearing and easy to get wrong:

* **Channel order is BGR.** Upstream reads images with ``cv2.imread`` and feeds the array straight to
  the network — raw 0-255 floats, no ``/255``, no mean/std normalisation (``ValTransform(legacy=False)``).
  Feeding RGB instead degrades detection, and on this variant it does so *quietly*: on this
  repository's own tutorial scene it simply loses the bench and leaves the other three objects'
  scores unchanged to three decimals. The smaller YOLOX-S sibling is much noisier about the same
  mistake, which makes this one easier to ship by accident.
* **The letterbox is reproduced with Pillow, not OpenCV.** Upstream resizes with
  ``cv2.INTER_LINEAR``; this package uses ``PIL.Image.BILINEAR`` so the dependency set stays at torch,
  torchvision, numpy and Pillow. The two kernels are not bit-identical in general, but the resize is
  skipped outright when the input is already ``INPUT_SIZE`` (verified: a Pillow bilinear resize to the
  source size is the identity), so the default 640x640 sample path is unaffected. Only a BYOD image of
  some other size goes through the differing kernel; ``docs/WEIGHTS.md`` records this.
"""

from __future__ import annotations

import hashlib
import json
import math
import urllib.request
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import TYPE_CHECKING, Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .coco_classes import COCO_CLASSES` removed — names are kernel globals defined by the carried modules

if TYPE_CHECKING:  # pragma: no cover - typing only, never executed
    pass

# --- Identity ---------------------------------------------------------------------------------
# YOLOX has no Hugging Face model repository. MODEL_ID names the upstream project and MODEL_REVISION
# the immutable GitHub release tag whose assets carry every published checkpoint (0.2.0 and 0.3.0 ship
# no weights of their own; upstream's own `yolox/models/build.py` at 0.3.0 points its download URLs
# back at 0.1.1rc0, which is why the 0.3.0 code below pairs with a 0.1.1rc0 checkpoint).
MODEL_ID = "Megvii-BaseDetection/YOLOX"
MODEL_REVISION = "0.1.1rc0"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "yolox-x"
# The upstream commit the vendored model code was taken from; see docs/UPSTREAM.md.
UPSTREAM_CODE_REVISION = "419778480ab6ec0590e5d3831b3afb3b46ab2aa3"
CHECKPOINT_FILE = "yolox_x.pth"
RELEASE_ASSET_URL = (
    f"https://github.com/Megvii-BaseDetection/YOLOX/releases/download/{MODEL_REVISION}/{CHECKPOINT_FILE}"
)
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# --- Architecture -----------------------------------------------------------------------------
# YOLOX-X scaling factors, from upstream `exps/default/yolox_x.py`. YOLOX-X is 1.33/1.25 and lives in
# its own repository; these two constants are the only architectural difference between the variants.
MODEL_DEPTH = 1.33
MODEL_WIDTH = 1.25
# `in_channels` and `act` are upstream `Exp` defaults (yolox/exp/yolox_base.py).
BACKBONE_CHANNELS = (256, 512, 1024)
ACTIVATION = "silu"
# BatchNorm overrides applied by upstream `Exp.get_model`'s `init_yolo`.
BN_EPS = 1e-3
BN_MOMENTUM = 0.03
# Upstream `Exp.get_model` calls `head.initialize_biases(1e-2)`: the focal-style prior that starts a
# fresh head at a 1% objectness/class probability instead of 50%.
HEAD_PRIOR_PROB = 1e-2
# Upstream `Exp.test_size`. The letterbox target; any multiple of 32 is architecturally valid.
INPUT_SIZE = (640, 640)
# Upstream `preproc` fills the unused part of the canvas with this grey rather than black.
PAD_VALUE = 114
# The three head strides produce 80x80 + 40x40 + 20x20 = 8400 anchor points at 640x640, and the head
# emits exactly one prediction per point, so nothing can survive NMS beyond this many boxes.
MAX_DETECTIONS = (INPUT_SIZE[0] // 8) ** 2 + (INPUT_SIZE[0] // 16) ** 2 + (INPUT_SIZE[0] // 32) ** 2

# --- Labels -----------------------------------------------------------------------------------
# The 80 COCO 2017 classes in upstream `yolox/data/datasets/coco_classes.py` order, which is the order
# the checkpoint's classification head was trained in.
LABELS: tuple[str, ...] = tuple(COCO_CLASSES)

# --- Thresholds -------------------------------------------------------------------------------
# Upstream ships two threshold pairs for two different purposes and this package keeps both named
# rather than averaging them into one house default:
#   * the demo pair (tools/demo.py --conf 0.3 --nms 0.3), meant for looking at pictures, and
#   * the evaluation pair (Exp.test_conf 0.01 / Exp.nmsthre 0.65), meant for COCO mAP, which keeps
#     far more low-scoring boxes because average precision rewards recall.
# Neither is a calibration for any deployment; the caller owns the choice and passes it explicitly.
DETECTION_THRESHOLD = 0.3
NMS_THRESHOLD = 0.3
EVAL_DETECTION_THRESHOLD = 0.01
EVAL_NMS_THRESHOLD = 0.65

# --- Input ceilings ---------------------------------------------------------------------------
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
# Upstream `TrainTransform(max_labels=50)`; the head's label tensor is zero-padded to this width.
MAX_LABELS_PER_IMAGE = 50

# --- Fine-tuning ceilings ---------------------------------------------------------------------
# Bounded so a tutorial adaptation cannot turn into an unbounded training run. These are this
# package's own operating limits, not upstream training recipe values: upstream trains YOLOX-X for
# 300 epochs on COCO with mosaic/mixup augmentation, which is out of scope here.
MAX_TRAIN_IMAGES = 200
MAX_EPOCHS = 20
MAX_CLASSES = 80
# Chosen by measurement, not taste — but the grid was run on the sibling YOLOX-S row, not here: it
# put 6 epochs at 1e-3 with a frozen backbone at held-out AP50 1.0 against 0.355 at 3 epochs, and a
# total collapse to 0.0 for an unfrozen full fine-tune at the same learning rate. This repository
# verified only that the chosen configuration works on YOLOX-X (AP50 0.0384 -> 1.0000); a full
# fine-tune of X was never run. docs/release-verification.md says so explicitly.
DEFAULT_EPOCHS = 6
DEFAULT_BATCH_SIZE = 2
DEFAULT_LEARNING_RATE = 1e-3
DEFAULT_SEED = 0
# Artifact format tag written into every exported artifact and checked on reload.
ARTIFACT_FORMAT = "dimer-yolox-detection-adapter/1"
ARTIFACT_FILE = "yolox-x-detection-adapter.pt"
# IoU thresholds of the COCO primary metric: AP@[.50:.95] averaged over ten thresholds.
COCO_IOU_THRESHOLDS = tuple(round(0.50 + 0.05 * i, 2) for i in range(10))
# COCO scores at most this many detections per image. Without the cap a model that has not learned to
# suppress background can post thousands of boxes per image at the 0.01 evaluation threshold, which
# both inflates its own recall tail and makes the matching quadratic in junk.
MAX_EVAL_DETECTIONS = 100


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


# --- Snapshot ---------------------------------------------------------------------------------


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch.

    The checkpoint is a pickle, so this runs *before* anything unpickles it: ``from_pretrained``
    verifies, then imports torch, then loads.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _release_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file from the pinned GitHub release asset URL.

    There is no Hugging Face repository to resolve and no ``main`` to drift onto: a release tag's
    assets are immutable, and the manifest digest is re-checked by ``verify_snapshot`` afterwards, so
    a substituted asset is caught before anything is unpickled.
    """
    url = f"https://github.com/{MODEL_ID}/releases/download/{MODEL_REVISION}/{relative_path}"
    root.mkdir(parents=True, exist_ok=True)
    target = root / relative_path
    tmp = target.with_suffix(target.suffix + ".part")
    with urllib.request.urlopen(url, timeout=120) as response, open(tmp, "wb") as fh:  # noqa: S310
        while chunk := response.read(1 << 20):
            fh.write(chunk)
    tmp.replace(target)


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally; return the relative paths fetched.

    A fresh clone commits the manifest but git-ignores the checkpoint, so this is the normal path.
    ``verify_snapshot`` still runs afterwards and is what makes the download trustworthy.
    """
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them from the {MODEL_REVISION} release assets"
        )
    fetch = downloader or _release_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


# --- Geometry and preprocessing ---------------------------------------------------------------


def box_iou(a: Sequence[float], b: Sequence[float]) -> float:
    """Intersection-over-union of two xyxy pixel boxes; the building block of the AP computation."""
    if len(a) != 4 or len(b) != 4:
        raise ValueError("boxes must be [x0, y0, x1, y1]")
    if a[2] < a[0] or a[3] < a[1] or b[2] < b[0] or b[3] < b[1]:
        raise ValueError("boxes must satisfy x0 <= x1 and y0 <= y1")
    inter_w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    inter_h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = inter_w * inter_h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / union) if union > 0 else 0.0


def preprocess(image: Image.Image) -> tuple[np.ndarray, float]:
    """Upstream's ``preproc`` letterbox, in Pillow and in BGR: returns (CHW float32 array, ratio).

    The array is raw 0-255 floats in **BGR** channel order on a ``PAD_VALUE`` canvas with the image
    pasted at the top-left corner and the aspect ratio preserved — exactly what upstream feeds the
    network. Divide the returned box coordinates by ``ratio`` to map predictions back to input pixels.
    """
    width, height = image.size
    ratio = min(INPUT_SIZE[0] / height, INPUT_SIZE[1] / width)
    new_w, new_h = int(width * ratio), int(height * ratio)
    canvas = np.full((INPUT_SIZE[0], INPUT_SIZE[1], 3), PAD_VALUE, dtype=np.uint8)
    # Skipped when the input is already INPUT_SIZE, which is the default sample path: a Pillow
    # bilinear resize to the source size is the identity, so no resize kernel is involved at all.
    resized = image if (new_w, new_h) == (width, height) else image.resize((new_w, new_h), Image.BILINEAR)
    canvas[:new_h, :new_w] = np.asarray(resized, dtype=np.uint8)
    chw = canvas.transpose(2, 0, 1)[::-1]  # HWC RGB -> CHW BGR
    return np.ascontiguousarray(chw, dtype=np.float32), ratio


# --- Validation -------------------------------------------------------------------------------


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


def _check_threshold(value: Any, name: str = "threshold") -> float:
    if isinstance(value, bool) or not isinstance(value, int | float) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a number in [0, 1], got {value!r}")
    return float(value)


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB): a photograph or a rendered scene",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "threshold": [0.0, 1.0],
    "nms_threshold": [0.0, 1.0],
    "labels": list(LABELS),
    "max_detections": MAX_DETECTIONS,
    "preprocessing": (
        f"image converted to RGB, letterboxed onto a {INPUT_SIZE[0]}x{INPUT_SIZE[1]} canvas filled with "
        f"{PAD_VALUE} (aspect ratio preserved, pasted top-left), then reordered to BGR and kept as raw "
        "0-255 float32 with no rescaling and no mean/std normalisation, matching upstream "
        "ValTransform(legacy=False); returned boxes are divided by the letterbox ratio to land back in "
        "input pixels"
    ),
}

TRAIN_SCHEMA: dict[str, Any] = {
    "record": (
        "a mapping with 'image' (PIL.Image.Image), 'boxes' (list of xyxy pixel boxes in that image's "
        "own coordinates) and 'labels' (list of class names, one per box, drawn from class_names)"
    ),
    "images": [1, MAX_TRAIN_IMAGES],
    "labels_per_image": [0, MAX_LABELS_PER_IMAGE],
    "classes": [1, MAX_CLASSES],
    "epochs": [1, MAX_EPOCHS],
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "preprocessing": INPUT_SCHEMA["preprocessing"],
    "note": (
        "boxes are converted to the head's own target form — (class index, cx, cy, w, h) in letterboxed "
        f"canvas pixels, zero-padded to {MAX_LABELS_PER_IMAGE} rows per image — inside finetune; a caller "
        "supplies xyxy in input-image pixels and never touches that conversion"
    ),
}


def _check_inputs(image: Any, threshold: Any, nms_threshold: Any) -> tuple[Image.Image, float, float]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``detect`` and ``validate_inputs`` both route through this function so their acceptance criteria
    cannot diverge.
    """
    return (
        validate_image(image),
        _check_threshold(threshold, "threshold"),
        _check_threshold(nms_threshold, "nms_threshold"),
    )


def validate_inputs(
    image: Image.Image,
    *,
    threshold: float = DETECTION_THRESHOLD,
    nms_threshold: float = NMS_THRESHOLD,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``detect`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _rgb, checked, checked_nms = _check_inputs(image, threshold, nms_threshold)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (detect takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "threshold": checked,
        "nms_threshold": checked_nms,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    epochs: int = DEFAULT_EPOCHS,
) -> dict[str, Any]:
    """Validation stage for the adaptation path: return the dataset manifest, or raise.

    Applies exactly the ceilings ``finetune`` applies, so a dataset this accepts cannot be refused
    later. Boxes are checked in the record's own image pixels, before any letterboxing.
    """
    if not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise TypeError(f"records must be a sequence of mappings, got {type(records).__name__}")
    names = list(class_names)
    if not names:
        raise ValueError("class_names must name at least one class")
    if len(names) > MAX_CLASSES:
        raise ValueError(f"class_names has {len(names)} entries > MAX_CLASSES {MAX_CLASSES}")
    if len(set(names)) != len(names):
        raise ValueError("class_names must not repeat a name")
    if not all(isinstance(name, str) and name for name in names):
        raise ValueError("every class name must be a non-empty string")
    if not 1 <= len(records) <= MAX_TRAIN_IMAGES:
        raise ValueError(f"records has {len(records)} images, expected 1..{MAX_TRAIN_IMAGES}")
    if not isinstance(epochs, int) or isinstance(epochs, bool) or not 1 <= epochs <= MAX_EPOCHS:
        raise ValueError(f"epochs must be an int in 1..{MAX_EPOCHS}, got {epochs!r}")

    per_class = dict.fromkeys(names, 0)
    total_boxes = 0
    for index, record in enumerate(records):
        if not isinstance(record, Mapping):
            raise TypeError(f"record {index} must be a mapping, got {type(record).__name__}")
        missing = {"image", "boxes", "labels"} - set(record)
        if missing:
            raise ValueError(f"record {index} is missing {sorted(missing)}")
        image = validate_image(record["image"])
        boxes, labels = list(record["boxes"]), list(record["labels"])
        if len(boxes) != len(labels):
            raise ValueError(f"record {index}: {len(boxes)} boxes but {len(labels)} labels")
        if len(boxes) > MAX_LABELS_PER_IMAGE:
            raise ValueError(
                f"record {index}: {len(boxes)} boxes > MAX_LABELS_PER_IMAGE {MAX_LABELS_PER_IMAGE}"
            )
        width, height = image.size
        for box, label in zip(boxes, labels, strict=True):
            if len(box) != 4:
                raise ValueError(f"record {index}: box {box!r} is not [x0, y0, x1, y1]")
            x0, y0, x1, y1 = (float(v) for v in box)
            if not (x1 > x0 and y1 > y0):
                raise ValueError(f"record {index}: box {box!r} must satisfy x0 < x1 and y0 < y1")
            if not (x0 >= 0 and y0 >= 0 and x1 <= width and y1 <= height):
                raise ValueError(f"record {index}: box {box!r} falls outside the {width}x{height} image")
            if label not in per_class:
                raise ValueError(f"record {index}: label {label!r} is not in class_names {names}")
            per_class[label] += 1
            total_boxes += 1
    empty = [name for name, count in per_class.items() if count == 0]
    return {
        "schema": dict(TRAIN_SCHEMA),
        "class_names": names,
        "n_images": len(records),
        "n_boxes": total_boxes,
        "boxes_per_class": per_class,
        "epochs": epochs,
        "verdict": "accepted",
        "findings": (
            [f"classes with no boxes in this dataset: {empty}; their head outputs stay untrained"]
            if empty
            else []
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def split_records(
    records: Sequence[Mapping[str, Any]],
    *,
    holdout: float = 0.25,
    seed: int = DEFAULT_SEED,
) -> tuple[list[Mapping[str, Any]], list[Mapping[str, Any]]]:
    """Deterministic train/held-out split. The split is the caller's, not the model's: it happens
    before any weight is touched, and the held-out part is never shown to ``finetune``."""
    if not 0.0 < holdout < 1.0:
        raise ValueError(f"holdout must be in (0, 1), got {holdout!r}")
    order = np.random.default_rng(seed).permutation(len(records))
    n_holdout = max(1, int(round(len(records) * holdout)))
    if n_holdout >= len(records):
        raise ValueError(f"holdout {holdout} leaves no training images out of {len(records)}")
    held = [records[i] for i in order[:n_holdout]]
    train = [records[i] for i in order[n_holdout:]]
    return train, held


# --- Evaluation -------------------------------------------------------------------------------


def average_precision(
    predictions: Sequence[Sequence[Mapping[str, Any]]],
    references: Sequence[Mapping[str, Any]],
    class_names: Sequence[str],
    *,
    iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
) -> dict[str, Any]:
    """COCO-style average precision over a list of images.

    ``predictions[i]`` are the detections for image ``i`` (``label``, ``score``, ``box``) and
    ``references[i]`` carries that image's ``boxes``/``labels``. Matching is the COCO rule: within a
    class, detections in descending score order each claim the highest-IoU unclaimed reference above
    the threshold; unmatched detections are false positives and unmatched references false negatives.
    Precision is interpolated over 101 recall points, AP is averaged over classes that have at least
    one reference, and ``ap`` is the mean over ``iou_thresholds``.

    This is a faithful small implementation, not ``pycocotools``: it has no area ranges and no crowd
    handling, and it scores whatever list it is handed rather than applying the per-image detection
    cap itself (``evaluate`` applies ``MAX_EVAL_DETECTIONS`` before calling in). It must not be
    compared against published COCO numbers; it exists to score a tutorial's own held-out images.
    """
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} prediction lists but {len(references)} references")
    names = list(class_names)
    recall_points = np.linspace(0.0, 1.0, 101)
    per_threshold: dict[float, dict[str, float]] = {}
    for threshold in iou_thresholds:
        per_class: dict[str, float] = {}
        for name in names:
            scored: list[tuple[float, bool]] = []
            n_references = 0
            for dets, reference in zip(predictions, references, strict=True):
                ref_boxes = [
                    box
                    for box, label in zip(reference["boxes"], reference["labels"], strict=True)
                    if label == name
                ]
                n_references += len(ref_boxes)
                claimed = [False] * len(ref_boxes)
                candidates = sorted((d for d in dets if d["label"] == name), key=lambda d: -float(d["score"]))
                for det in candidates:
                    best, best_iou = -1, 0.0
                    for j, ref_box in enumerate(ref_boxes):
                        if claimed[j]:
                            continue
                        value = box_iou(det["box"], ref_box)
                        if value > best_iou:
                            best, best_iou = j, value
                    hit = best >= 0 and best_iou >= threshold
                    if hit:
                        claimed[best] = True
                    scored.append((float(det["score"]), hit))
            if n_references == 0:
                continue  # a class with no references contributes no AP, as in COCO
            scored.sort(key=lambda pair: -pair[0])
            true_positives = np.cumsum([1 if hit else 0 for _score, hit in scored])
            false_positives = np.cumsum([0 if hit else 1 for _score, hit in scored])
            if not scored:
                per_class[name] = 0.0
                continue
            recall = true_positives / n_references
            precision = true_positives / np.maximum(true_positives + false_positives, 1)
            # Monotone envelope, then sample at the 101 recall points (COCO's interpolation).
            precision = np.maximum.accumulate(precision[::-1])[::-1]
            sampled = np.zeros_like(recall_points)
            indices = np.searchsorted(recall, recall_points, side="left")
            valid = indices < len(precision)
            sampled[valid] = precision[indices[valid]]
            per_class[name] = float(sampled.mean())
        per_threshold[threshold] = per_class
    scored_classes = sorted({name for values in per_threshold.values() for name in values})
    means = {
        threshold: (float(np.mean(list(values.values()))) if values else 0.0)
        for threshold, values in per_threshold.items()
    }
    return {
        "ap": float(np.mean(list(means.values()))) if means else 0.0,
        "ap50": means.get(0.5, 0.0),
        "ap75": means.get(0.75, 0.0),
        "per_class_ap50": per_threshold.get(0.5, {}),
        "iou_thresholds": [float(t) for t in iou_thresholds],
        "scored_classes": scored_classes,
        "n_images": len(references),
        "n_references": sum(len(r["boxes"]) for r in references),
    }


def evaluation_report(
    result: Mapping[str, Any],
    ground_truth_boxes: Mapping[str, Sequence[Sequence[float]]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Single-image evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth_boxes`` (label -> xyxy reference boxes for one image) the report carries one
    ``box_iou`` entry per reference — the best-overlapping detection **of the same label** — as
    sample-sanity geometry evidence. Without them the verdict is ``not-measurable``. For a scored
    held-out set use ``average_precision`` instead; this helper deliberately does not call a
    single-image IoU a mean average precision.
    """
    detections = list(result["detections"])
    labels = tuple(result.get("class_names") or LABELS)
    base = {
        "task": f"object detection over {len(labels)} classes on one image",
        "decision_rule": (
            "each of the 8400 anchor points emits one box, one objectness logit and one logit per "
            "class; a prediction survives when objectness x class probability (both sigmoids, not a "
            "softmax over classes) reaches the confidence threshold and it wins per-class NMS at the "
            "NMS threshold. Neither threshold is calibrated for any deployment"
        ),
        "threshold": result.get("threshold", DETECTION_THRESHOLD),
        "nms_threshold": result.get("nms_threshold", NMS_THRESHOLD),
        "sample_kind": sample_kind,
        "n_detections": len(detections),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if not ground_truth_boxes:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth object boxes were supplied for the evaluated image",
            "needs": (
                "labelled boxes per class on your own images, scored with average_precision over a "
                "held-out set to obtain AP@[.50:.95] and AP50; a single image's box_iou values are "
                "geometry sanity evidence and no labelled image set ships with this repository"
            ),
        }
    metrics = []
    for label, boxes in ground_truth_boxes.items():
        if label not in labels:
            raise ValueError(f"unknown reference label {label!r}; expected one of {len(labels)} class names")
        same_label = [det for det in detections if det["label"] == label]
        for index, box in enumerate(boxes):
            ious = [box_iou(det["box"], box) for det in same_label]
            best = max(range(len(ious)), key=ious.__getitem__) if ious else None
            metrics.append(
                {
                    "id": "box_iou",
                    "reference": f"{label}-{index}",
                    "value": ious[best] if best is not None else 0.0,
                    "matched_score": same_label[best]["score"] if best is not None else None,
                    "n_detected_same_label": len(same_label),
                    "estimation": "one reference box per object on a single image, no dispersion estimate",
                }
            )
    return {
        **base,
        "metrics": metrics,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(metrics)} reference box(es) on one tutorial image; geometry sanity evidence, "
            "not a detection benchmark"
        ),
        "needs": (
            "a labelled image set from the deployment domain (cameras, scenes, object classes) for any "
            "average-precision claim; use average_precision on a held-out split for that"
        ),
    }


# --- Pipeline ---------------------------------------------------------------------------------


@dataclass
class YoloxXDetectionPipeline:
    """YOLOX-X detection and bounded detection fine-tuning over a verified local checkpoint."""

    model: Any
    device: str
    class_names: tuple[str, ...]
    source: str = "local-snapshot"
    base_state_digest: str | None = None
    adapted: bool = False
    reinitialised: tuple[str, ...] = field(default_factory=tuple)

    # -- construction --------------------------------------------------------------------------

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        class_names: Sequence[str] | None = None,
        seed: int = DEFAULT_SEED,
    ) -> YoloxXDetectionPipeline:
        """Build YOLOX-X and load the verified checkpoint.

        With ``class_names`` the classification branch is rebuilt for that vocabulary and randomly
        initialised while the backbone, neck, box and objectness weights are kept — the starting point
        for ``finetune``. Without it the model keeps the checkpoint's own 80 COCO classes and every
        tensor is loaded ``strict=True``.

        ``seed`` fixes that random initialisation. It matters: the re-headed classification layers are
        the only untrained weights in the model, and they are what the pre-adaptation baseline is
        measuring, so leaving them at whatever the ambient global RNG happened to hold makes the
        baseline number irreproducible from run to run. The global RNG state is restored afterwards so
        seeding here cannot perturb a caller's own stream.
        """
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
        else:
            raise FileNotFoundError(
                f"no manifest at {root}; commit weights/{MODEL_KEY}/{MANIFEST_NAME} and stage "
                f"{CHECKPOINT_FILE} from the {MODEL_REVISION} release assets"
            )
        names = tuple(class_names) if class_names is not None else LABELS
        if not 1 <= len(names) <= MAX_CLASSES:
            raise ValueError(f"class_names must hold 1..{MAX_CLASSES} names, got {len(names)}")

        # Refuse an invalid snapshot before importing torch or unpickling anything.
        import torch

        checkpoint_path = root / CHECKPOINT_FILE
        # The upstream checkpoint is a pickle, not SafeTensors. It is a plain tensor/int dict, so
        # weights_only=True is enough to load it without allowing arbitrary globals; the digest was
        # already checked above. Upstream's own loader passes neither.
        payload = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        state = payload["model"] if isinstance(payload, dict) and "model" in payload else payload
        digest = hashlib.sha256(
            b"".join(state[key].detach().cpu().numpy().tobytes() for key in sorted(state))
        ).hexdigest()

        rng_state = torch.get_rng_state()
        try:
            torch.manual_seed(seed)
            model = cls._build(len(names))
        finally:
            torch.set_rng_state(rng_state)
        reinitialised: tuple[str, ...] = ()
        if len(names) == len(LABELS) and class_names is None:
            model.load_state_dict(state, strict=True)
        else:
            # Transfer learning: keep everything whose shape still matches (backbone, PAFPN, the box
            # and objectness heads) and leave the class-conditional tensors at their initialisation.
            own = model.state_dict()
            transferable = {k: v for k, v in state.items() if k in own and own[k].shape == v.shape}
            dropped = tuple(sorted(set(own) - set(transferable)))
            model.load_state_dict(transferable, strict=False)
            reinitialised = dropped
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(resolved_device).eval()
        return cls(
            model=model,
            device=resolved_device,
            class_names=names,
            base_state_digest=digest,
            reinitialised=reinitialised,
        )

    @staticmethod
    def _build(num_classes: int) -> Any:
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .yolo_head import YOLOXHead` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .yolo_pafpn import YOLOPAFPN` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .yolox import YOLOX` removed — names are kernel globals defined by the carried modules

        channels = list(BACKBONE_CHANNELS)
        backbone = YOLOPAFPN(MODEL_DEPTH, MODEL_WIDTH, in_channels=channels, act=ACTIVATION)
        head = YOLOXHead(num_classes, MODEL_WIDTH, in_channels=channels, act=ACTIVATION)
        model = YOLOX(backbone, head)
        # Upstream `init_yolo`: the BatchNorm defaults YOLOX trains with, not PyTorch's.
        for module in model.modules():
            if isinstance(module, torch.nn.BatchNorm2d):
                module.eps = BN_EPS
                module.momentum = BN_MOMENTUM
        # Upstream `Exp.get_model` also does this, and it matters: it sets every classification and
        # objectness bias to -log((1 - p) / p) for p = HEAD_PRIOR_PROB, so a freshly initialised head
        # starts by predicting "almost certainly nothing here" instead of a coin flip at all 8400
        # anchors. Omitting it lets the objectness term saturate within a few steps of a re-headed
        # fine-tune and the model collapses to one class at score 1.0 everywhere. For the COCO path
        # the checkpoint overwrites these biases immediately, so it is a no-op there.
        model.head.initialize_biases(HEAD_PRIOR_PROB)
        return model

    @classmethod
    def load_artifact(
        cls,
        path: str | Path,
        *,
        device: str | None = None,
    ) -> YoloxXDetectionPipeline:
        """Rebuild an adapted pipeline from an exported artifact, in a process that never fine-tuned.

        The artifact records its own format tag, pinned identity, model key, class vocabulary and the
        digest of the base weights it started from; a mismatch raises rather than silently loading a
        different model. The model key matters because every YOLOX variant is an asset of the same
        upstream release, so the id and revision alone do not say which network the weights are for.
        """
        import torch

        artifact_path = Path(path)
        payload = torch.load(artifact_path, map_location="cpu", weights_only=True)
        if payload.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {payload.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if payload.get("model_id") != MODEL_ID or payload.get("model_revision") != MODEL_REVISION:
            raise ValueError(
                f"artifact was built on {payload.get('model_id')}@{payload.get('model_revision')}, "
                f"package pins {MODEL_ID}@{MODEL_REVISION}"
            )
        # MODEL_KEY, not just the model id and revision: the YOLOX variants are all assets of the
        # same upstream release, so `Megvii-BaseDetection/YOLOX@0.1.1rc0` does not identify which
        # network an artifact belongs to. Without this, a sibling variant's adapter passes the
        # identity check and fails afterwards inside load_state_dict with a tensor-shape error.
        if payload.get("model_key") != MODEL_KEY:
            raise ValueError(
                f"artifact was built on the {payload.get('model_key')!r} variant, "
                f"package pins {MODEL_KEY!r}"
            )
        names = tuple(payload["class_names"])
        model = cls._build(len(names))
        model.load_state_dict(payload["state_dict"], strict=True)
        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        model = model.to(resolved_device).eval()
        return cls(
            model=model,
            device=resolved_device,
            class_names=names,
            source=f"artifact:{artifact_path.name}",
            base_state_digest=payload.get("base_state_digest"),
            adapted=True,
        )

    # -- inference -----------------------------------------------------------------------------

    def detect(
        self,
        image: Image.Image,
        *,
        threshold: float = DETECTION_THRESHOLD,
        nms_threshold: float = NMS_THRESHOLD,
    ) -> dict[str, Any]:
        """Detect objects on one image; boxes are xyxy pixel coordinates in the input image."""
        rgb, checked, checked_nms = _check_inputs(image, threshold, nms_threshold)
        detections = self._run(rgb, checked, checked_nms)
        if len(detections) > MAX_DETECTIONS:
            raise RuntimeError(
                f"backend returned {len(detections)} detections > anchor count {MAX_DETECTIONS}"
            )
        for det in detections:
            if (
                set(det) != {"box", "label", "score"}
                or len(det["box"]) != 4
                or det["label"] not in self.class_names
            ):
                raise RuntimeError(f"backend returned a malformed detection: {det!r}")
        return {
            "detections": sorted(detections, key=lambda d: -d["score"]),
            "threshold": checked,
            "nms_threshold": checked_nms,
            "width": rgb.width,
            "height": rgb.height,
            "class_names": list(self.class_names),
            "adapted": self.adapted,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def _run(self, image: Image.Image, threshold: float, nms_threshold: float) -> list[dict[str, Any]]:
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .ops import postprocess` removed — names are kernel globals defined by the carried modules

        chw, ratio = preprocess(image)
        tensor = torch.from_numpy(chw).unsqueeze(0).to(self.device)
        was_training = self.model.training
        self.model.eval()
        # no_grad rather than inference_mode: upstream `postprocess` writes the xyxy conversion back
        # into its argument in place, which an inference-mode tensor refuses. Upstream's own
        # tools/demo.py uses no_grad for the same reason.
        with torch.no_grad():
            raw = self.model(tensor)
        kept = postprocess(
            raw, len(self.class_names), conf_thre=threshold, nms_thre=nms_threshold, class_agnostic=False
        )[0]
        if was_training:
            self.model.train()
        if kept is None:
            return []
        detections = []
        for x0, y0, x1, y1, objectness, class_conf, class_id in kept.tolist():
            detections.append(
                {
                    "box": [x0 / ratio, y0 / ratio, x1 / ratio, y1 / ratio],
                    "label": self.class_names[int(class_id)],
                    "score": float(objectness * class_conf),
                }
            )
        return detections

    def detect_many(
        self,
        images: Sequence[Image.Image],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
        nms_threshold: float = EVAL_NMS_THRESHOLD,
    ) -> list[list[dict[str, Any]]]:
        """Detections for several images, one ``detect`` per image, defaulting to the *evaluation*
        thresholds — average precision rewards recall, so scoring at the demo thresholds understates
        it. Batching is deliberately not implemented: see the model card's out-of-scope section."""
        return [
            self.detect(image, threshold=threshold, nms_threshold=nms_threshold)["detections"]
            for image in images
        ]

    # -- adaptation ----------------------------------------------------------------------------

    def finetune(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        epochs: int = DEFAULT_EPOCHS,
        batch_size: int = DEFAULT_BATCH_SIZE,
        learning_rate: float = DEFAULT_LEARNING_RATE,
        seed: int = DEFAULT_SEED,
        freeze_backbone: bool = True,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded SimOTA fine-tune on ``records``; returns the run manifest and per-epoch losses.

        The loss and the label assignment are upstream's: the vendored ``YOLOXHead`` in training mode
        computes SimOTA matching, the IoU/objectness/classification terms and their weighting. What
        this method owns is the bounded loop around it — the target-tensor conversion, batching, the
        optimiser, the seed and the ceilings. ``use_l1`` stays off: upstream only enables the extra L1
        box term for the last 15 of 300 epochs, and switching it on for a 3-epoch tutorial would
        change the loss scale for no benefit.

        ``freeze_backbone`` (on by default) trains the detection head only and leaves the CSPDarknet
        backbone and PAFPN neck exactly as the COCO checkpoint left them. That is the honest default
        for a bounded tutorial on CPU: it is several times faster per step, it cannot damage the
        pretrained features with a handful of gradient steps on a small set, and the features a
        COCO-trained backbone already has are what makes a few-epoch adaptation work at all. Pass
        ``freeze_backbone=False`` for a full fine-tune when you have the data and the compute for it.

        Mutates this pipeline in place (``adapted`` becomes True) and leaves the model in eval mode.
        """
        import torch

        manifest = validate_dataset(records, self.class_names, epochs=epochs)
        if not isinstance(batch_size, int) or isinstance(batch_size, bool) or batch_size < 1:
            raise ValueError(f"batch_size must be a positive int, got {batch_size!r}")
        if not isinstance(learning_rate, int | float) or isinstance(learning_rate, bool):
            raise ValueError(f"learning_rate must be a number, got {learning_rate!r}")
        if not 0.0 < float(learning_rate) <= 1.0:
            raise ValueError(f"learning_rate must be in (0, 1], got {learning_rate!r}")

        torch.manual_seed(seed)
        rng = np.random.default_rng(seed)
        images, targets = self._as_batch_tensors(records)
        images, targets = images.to(self.device), targets.to(self.device)

        for parameter in self.model.backbone.parameters():
            parameter.requires_grad = not freeze_backbone
        self.model.train()
        if freeze_backbone:
            # Keep the frozen BatchNorm statistics the checkpoint was trained with: in train() mode a
            # BatchNorm updates its running mean/var even with requires_grad=False, so 27 steps of a
            # tutorial batch would quietly rewrite the backbone's normalisation.
            self.model.backbone.eval()
        self.model.head.use_l1 = False
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = torch.optim.SGD(trainable, lr=float(learning_rate), momentum=0.9, weight_decay=5e-4)
        history: list[dict[str, Any]] = []
        n = len(records)
        for epoch in range(epochs):
            order = rng.permutation(n)
            epoch_losses: list[dict[str, float]] = []
            for start in range(0, n, batch_size):
                index = torch.as_tensor(order[start : start + batch_size].copy(), dtype=torch.long)
                optimizer.zero_grad(set_to_none=True)
                losses = self.model(images[index].to(self.device), targets[index].to(self.device))
                losses["total_loss"].backward()
                optimizer.step()
                epoch_losses.append(
                    {
                        key: float(value.detach()) if torch.is_tensor(value) else float(value)
                        for key, value in losses.items()
                    }
                )
            row = {
                "epoch": epoch + 1,
                "steps": len(epoch_losses),
                **{
                    key: round(float(np.mean([loss[key] for loss in epoch_losses])), 5)
                    for key in epoch_losses[0]
                },
            }
            if not math.isfinite(row["total_loss"]):
                raise RuntimeError(f"epoch {epoch + 1} produced a non-finite loss: {row}")
            history.append(row)
            if progress is not None:
                progress(row)
        self.model.eval()
        self.adapted = True
        return {
            "dataset": manifest,
            "history": history,
            "epochs": epochs,
            "batch_size": batch_size,
            "learning_rate": float(learning_rate),
            "seed": seed,
            "optimizer": "SGD(momentum=0.9, weight_decay=5e-4)",
            "freeze_backbone": freeze_backbone,
            "trainable_parameters": sum(p.numel() for p in trainable),
            "total_parameters": sum(p.numel() for p in self.model.parameters()),
            "use_l1": False,
            "loss": "upstream YOLOXHead SimOTA (IoU + objectness + classification)",
            "device": self.device,
            "class_names": list(self.class_names),
            "reinitialised_tensors": list(self.reinitialised),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def _as_batch_tensors(self, records: Sequence[Mapping[str, Any]]) -> tuple[Any, Any]:
        """Letterbox every record and build the head's own label tensor.

        The head reads ``labels[b, :, 0]`` as a class index and ``labels[b, :, 1:5]`` as (cx, cy, w, h)
        in letterboxed-canvas pixels, counting valid rows as those whose five values do not sum to
        zero — so rows must start at index 0 and stay contiguous.
        """
        import torch

        batch = torch.zeros(len(records), 3, INPUT_SIZE[0], INPUT_SIZE[1])
        labels = torch.zeros(len(records), MAX_LABELS_PER_IMAGE, 5)
        for index, record in enumerate(records):
            chw, ratio = preprocess(validate_image(record["image"]))
            batch[index] = torch.from_numpy(chw)
            for row, (box, label) in enumerate(zip(record["boxes"], record["labels"], strict=True)):
                x0, y0, x1, y1 = (float(v) * ratio for v in box)
                labels[index, row] = torch.tensor(
                    [
                        float(self.class_names.index(label)),
                        (x0 + x1) / 2,
                        (y0 + y1) / 2,
                        x1 - x0,
                        y1 - y0,
                    ]
                )
        return batch, labels

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        threshold: float = EVAL_DETECTION_THRESHOLD,
        nms_threshold: float = EVAL_NMS_THRESHOLD,
        iou_thresholds: Sequence[float] = COCO_IOU_THRESHOLDS,
        max_detections: int = MAX_EVAL_DETECTIONS,
    ) -> dict[str, Any]:
        """Score a held-out set: average precision plus the request that produced the detections."""
        predictions = self.detect_many(
            [record["image"] for record in records], threshold=threshold, nms_threshold=nms_threshold
        )
        raw_counts = [len(dets) for dets in predictions]
        # COCO's per-image cap, applied to the score-ordered detections before matching.
        predictions = [dets[:max_detections] for dets in predictions]
        metrics = average_precision(predictions, records, self.class_names, iou_thresholds=iou_thresholds)
        return {
            **metrics,
            "threshold": _check_threshold(threshold, "threshold"),
            "nms_threshold": _check_threshold(nms_threshold, "nms_threshold"),
            "max_detections": max_detections,
            "detections_before_cap": raw_counts,
            "adapted": self.adapted,
            "class_names": list(self.class_names),
            "implementation": (
                "package-local average_precision: COCO matching and 101-point interpolation, without "
                "pycocotools' area ranges, detection cap or crowd handling — not comparable to "
                "published COCO numbers"
            ),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # -- artifact ------------------------------------------------------------------------------

    def save_artifact(self, path: str | Path, *, notes: str | None = None) -> dict[str, Any]:
        """Write the adapted weights and their provenance as one artifact; return its descriptor.

        A single ``torch.save`` of tensors and plain metadata — no archive, no pickled objects beyond
        what ``weights_only=True`` accepts on reload, so there is no extraction step to make safe.
        """
        import torch

        artifact_path = Path(path)
        artifact_path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "format": ARTIFACT_FORMAT,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "upstream_code_revision": UPSTREAM_CODE_REVISION,
            "model_key": MODEL_KEY,
            "class_names": list(self.class_names),
            "depth": MODEL_DEPTH,
            "width": MODEL_WIDTH,
            "input_size": list(INPUT_SIZE),
            "adapted": self.adapted,
            "base_state_digest": self.base_state_digest,
            "notes": notes or "",
            "state_dict": {k: v.detach().cpu() for k, v in self.model.state_dict().items()},
        }
        torch.save(payload, artifact_path)
        return {
            "path": str(artifact_path),
            "bytes": artifact_path.stat().st_size,
            "sha256": _sha256(artifact_path),
            "format": ARTIFACT_FORMAT,
            "class_names": list(self.class_names),
            "tensors": len(payload["state_dict"]),
            "base_state_digest": self.base_state_digest,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

**Module 8/10:** `src/yolox_x_detection_pipeline/yolo_head.py` (carried verbatim; see the note above)

In [ ]:
#!/usr/bin/env python3
# -*- coding:utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

import math
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F

# standalone rewrite (build_notebook.py): `from .ops import bboxes_iou, meshgrid` removed — names are kernel globals defined by the carried modules

# standalone rewrite (build_notebook.py): `from .losses import IOUloss` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .network_blocks import BaseConv, DWConv` removed — names are kernel globals defined by the carried modules


class YOLOXHead(nn.Module):
    def __init__(
        self,
        num_classes,
        width=1.0,
        strides=[8, 16, 32],
        in_channels=[256, 512, 1024],
        act="silu",
        depthwise=False,
    ):
        """
        Args:
            act (str): activation type of conv. Defalut value: "silu".
            depthwise (bool): whether apply depthwise conv in conv branch. Defalut value: False.
        """
        super().__init__()

        self.n_anchors = 1
        self.num_classes = num_classes
        self.decode_in_inference = True  # for deploy, set to False

        self.cls_convs = nn.ModuleList()
        self.reg_convs = nn.ModuleList()
        self.cls_preds = nn.ModuleList()
        self.reg_preds = nn.ModuleList()
        self.obj_preds = nn.ModuleList()
        self.stems = nn.ModuleList()
        Conv = DWConv if depthwise else BaseConv

        for i in range(len(in_channels)):
            self.stems.append(
                BaseConv(
                    in_channels=int(in_channels[i] * width),
                    out_channels=int(256 * width),
                    ksize=1,
                    stride=1,
                    act=act,
                )
            )
            self.cls_convs.append(
                nn.Sequential(
                    *[
                        Conv(
                            in_channels=int(256 * width),
                            out_channels=int(256 * width),
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                        Conv(
                            in_channels=int(256 * width),
                            out_channels=int(256 * width),
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                    ]
                )
            )
            self.reg_convs.append(
                nn.Sequential(
                    *[
                        Conv(
                            in_channels=int(256 * width),
                            out_channels=int(256 * width),
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                        Conv(
                            in_channels=int(256 * width),
                            out_channels=int(256 * width),
                            ksize=3,
                            stride=1,
                            act=act,
                        ),
                    ]
                )
            )
            self.cls_preds.append(
                nn.Conv2d(
                    in_channels=int(256 * width),
                    out_channels=self.n_anchors * self.num_classes,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                )
            )
            self.reg_preds.append(
                nn.Conv2d(
                    in_channels=int(256 * width),
                    out_channels=4,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                )
            )
            self.obj_preds.append(
                nn.Conv2d(
                    in_channels=int(256 * width),
                    out_channels=self.n_anchors * 1,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                )
            )

        self.use_l1 = False
        self.l1_loss = nn.L1Loss(reduction="none")
        self.bcewithlog_loss = nn.BCEWithLogitsLoss(reduction="none")
        self.iou_loss = IOUloss(reduction="none")
        self.strides = strides
        self.grids = [torch.zeros(1)] * len(in_channels)

    def initialize_biases(self, prior_prob):
        for conv in self.cls_preds:
            b = conv.bias.view(self.n_anchors, -1)
            b.data.fill_(-math.log((1 - prior_prob) / prior_prob))
            conv.bias = torch.nn.Parameter(b.view(-1), requires_grad=True)

        for conv in self.obj_preds:
            b = conv.bias.view(self.n_anchors, -1)
            b.data.fill_(-math.log((1 - prior_prob) / prior_prob))
            conv.bias = torch.nn.Parameter(b.view(-1), requires_grad=True)

    def forward(self, xin, labels=None, imgs=None):
        outputs = []
        origin_preds = []
        x_shifts = []
        y_shifts = []
        expanded_strides = []

        for k, (cls_conv, reg_conv, stride_this_level, x) in enumerate(
            zip(self.cls_convs, self.reg_convs, self.strides, xin)
        ):
            x = self.stems[k](x)
            cls_x = x
            reg_x = x

            cls_feat = cls_conv(cls_x)
            cls_output = self.cls_preds[k](cls_feat)

            reg_feat = reg_conv(reg_x)
            reg_output = self.reg_preds[k](reg_feat)
            obj_output = self.obj_preds[k](reg_feat)

            if self.training:
                output = torch.cat([reg_output, obj_output, cls_output], 1)
                output, grid = self.get_output_and_grid(
                    output, k, stride_this_level, xin[0].type()
                )
                x_shifts.append(grid[:, :, 0])
                y_shifts.append(grid[:, :, 1])
                expanded_strides.append(
                    torch.zeros(1, grid.shape[1])
                    .fill_(stride_this_level)
                    .type_as(xin[0])
                )
                if self.use_l1:
                    batch_size = reg_output.shape[0]
                    hsize, wsize = reg_output.shape[-2:]
                    reg_output = reg_output.view(
                        batch_size, self.n_anchors, 4, hsize, wsize
                    )
                    reg_output = reg_output.permute(0, 1, 3, 4, 2).reshape(
                        batch_size, -1, 4
                    )
                    origin_preds.append(reg_output.clone())

            else:
                output = torch.cat(
                    [reg_output, obj_output.sigmoid(), cls_output.sigmoid()], 1
                )

            outputs.append(output)

        if self.training:
            return self.get_losses(
                imgs,
                x_shifts,
                y_shifts,
                expanded_strides,
                labels,
                torch.cat(outputs, 1),
                origin_preds,
                dtype=xin[0].dtype,
            )
        else:
            self.hw = [x.shape[-2:] for x in outputs]
            # [batch, n_anchors_all, 85]
            outputs = torch.cat(
                [x.flatten(start_dim=2) for x in outputs], dim=2
            ).permute(0, 2, 1)
            if self.decode_in_inference:
                return self.decode_outputs(outputs, dtype=xin[0].type())
            else:
                return outputs

    def get_output_and_grid(self, output, k, stride, dtype):
        grid = self.grids[k]

        batch_size = output.shape[0]
        n_ch = 5 + self.num_classes
        hsize, wsize = output.shape[-2:]
        if grid.shape[2:4] != output.shape[2:4]:
            yv, xv = meshgrid([torch.arange(hsize), torch.arange(wsize)])
            grid = torch.stack((xv, yv), 2).view(1, 1, hsize, wsize, 2).type(dtype)
            self.grids[k] = grid

        output = output.view(batch_size, self.n_anchors, n_ch, hsize, wsize)
        output = output.permute(0, 1, 3, 4, 2).reshape(
            batch_size, self.n_anchors * hsize * wsize, -1
        )
        grid = grid.view(1, -1, 2)
        output[..., :2] = (output[..., :2] + grid) * stride
        output[..., 2:4] = torch.exp(output[..., 2:4]) * stride
        return output, grid

    def decode_outputs(self, outputs, dtype):
        grids = []
        strides = []
        for (hsize, wsize), stride in zip(self.hw, self.strides):
            yv, xv = meshgrid([torch.arange(hsize), torch.arange(wsize)])
            grid = torch.stack((xv, yv), 2).view(1, -1, 2)
            grids.append(grid)
            shape = grid.shape[:2]
            strides.append(torch.full((*shape, 1), stride))

        grids = torch.cat(grids, dim=1).type(dtype)
        strides = torch.cat(strides, dim=1).type(dtype)

        outputs[..., :2] = (outputs[..., :2] + grids) * strides
        outputs[..., 2:4] = torch.exp(outputs[..., 2:4]) * strides
        return outputs

    def get_losses(
        self,
        imgs,
        x_shifts,
        y_shifts,
        expanded_strides,
        labels,
        outputs,
        origin_preds,
        dtype,
    ):
        bbox_preds = outputs[:, :, :4]  # [batch, n_anchors_all, 4]
        obj_preds = outputs[:, :, 4].unsqueeze(-1)  # [batch, n_anchors_all, 1]
        cls_preds = outputs[:, :, 5:]  # [batch, n_anchors_all, n_cls]

        # calculate targets
        nlabel = (labels.sum(dim=2) > 0).sum(dim=1)  # number of objects

        total_num_anchors = outputs.shape[1]
        x_shifts = torch.cat(x_shifts, 1)  # [1, n_anchors_all]
        y_shifts = torch.cat(y_shifts, 1)  # [1, n_anchors_all]
        expanded_strides = torch.cat(expanded_strides, 1)
        if self.use_l1:
            origin_preds = torch.cat(origin_preds, 1)

        cls_targets = []
        reg_targets = []
        l1_targets = []
        obj_targets = []
        fg_masks = []

        num_fg = 0.0
        num_gts = 0.0

        for batch_idx in range(outputs.shape[0]):
            num_gt = int(nlabel[batch_idx])
            num_gts += num_gt
            if num_gt == 0:
                cls_target = outputs.new_zeros((0, self.num_classes))
                reg_target = outputs.new_zeros((0, 4))
                l1_target = outputs.new_zeros((0, 4))
                obj_target = outputs.new_zeros((total_num_anchors, 1))
                fg_mask = outputs.new_zeros(total_num_anchors).bool()
            else:
                gt_bboxes_per_image = labels[batch_idx, :num_gt, 1:5]
                gt_classes = labels[batch_idx, :num_gt, 0]
                bboxes_preds_per_image = bbox_preds[batch_idx]

                try:
                    (
                        gt_matched_classes,
                        fg_mask,
                        pred_ious_this_matching,
                        matched_gt_inds,
                        num_fg_img,
                    ) = self.get_assignments(  # noqa
                        batch_idx,
                        num_gt,
                        total_num_anchors,
                        gt_bboxes_per_image,
                        gt_classes,
                        bboxes_preds_per_image,
                        expanded_strides,
                        x_shifts,
                        y_shifts,
                        cls_preds,
                        bbox_preds,
                        obj_preds,
                        labels,
                        imgs,
                    )
                except RuntimeError as e:
                    # TODO: the string might change, consider a better way
                    if "CUDA out of memory. " not in str(e):
                        raise  # RuntimeError might not caused by CUDA OOM

                    warnings.warn(
                        "OOM RuntimeError is raised due to the huge memory cost during label assignment. \
                           CPU mode is applied in this batch. If you want to avoid this issue, \
                           try to reduce the batch size or image size."
                    )
                    torch.cuda.empty_cache()
                    (
                        gt_matched_classes,
                        fg_mask,
                        pred_ious_this_matching,
                        matched_gt_inds,
                        num_fg_img,
                    ) = self.get_assignments(  # noqa
                        batch_idx,
                        num_gt,
                        total_num_anchors,
                        gt_bboxes_per_image,
                        gt_classes,
                        bboxes_preds_per_image,
                        expanded_strides,
                        x_shifts,
                        y_shifts,
                        cls_preds,
                        bbox_preds,
                        obj_preds,
                        labels,
                        imgs,
                        "cpu",
                    )

                torch.cuda.empty_cache()
                num_fg += num_fg_img

                cls_target = F.one_hot(
                    gt_matched_classes.to(torch.int64), self.num_classes
                ) * pred_ious_this_matching.unsqueeze(-1)
                obj_target = fg_mask.unsqueeze(-1)
                reg_target = gt_bboxes_per_image[matched_gt_inds]
                if self.use_l1:
                    l1_target = self.get_l1_target(
                        outputs.new_zeros((num_fg_img, 4)),
                        gt_bboxes_per_image[matched_gt_inds],
                        expanded_strides[0][fg_mask],
                        x_shifts=x_shifts[0][fg_mask],
                        y_shifts=y_shifts[0][fg_mask],
                    )

            cls_targets.append(cls_target)
            reg_targets.append(reg_target)
            obj_targets.append(obj_target.to(dtype))
            fg_masks.append(fg_mask)
            if self.use_l1:
                l1_targets.append(l1_target)

        cls_targets = torch.cat(cls_targets, 0)
        reg_targets = torch.cat(reg_targets, 0)
        obj_targets = torch.cat(obj_targets, 0)
        fg_masks = torch.cat(fg_masks, 0)
        if self.use_l1:
            l1_targets = torch.cat(l1_targets, 0)

        num_fg = max(num_fg, 1)
        loss_iou = (
            self.iou_loss(bbox_preds.view(-1, 4)[fg_masks], reg_targets)
        ).sum() / num_fg
        loss_obj = (
            self.bcewithlog_loss(obj_preds.view(-1, 1), obj_targets)
        ).sum() / num_fg
        loss_cls = (
            self.bcewithlog_loss(
                cls_preds.view(-1, self.num_classes)[fg_masks], cls_targets
            )
        ).sum() / num_fg
        if self.use_l1:
            loss_l1 = (
                self.l1_loss(origin_preds.view(-1, 4)[fg_masks], l1_targets)
            ).sum() / num_fg
        else:
            loss_l1 = 0.0

        reg_weight = 5.0
        loss = reg_weight * loss_iou + loss_obj + loss_cls + loss_l1

        return (
            loss,
            reg_weight * loss_iou,
            loss_obj,
            loss_cls,
            loss_l1,
            num_fg / max(num_gts, 1),
        )

    def get_l1_target(self, l1_target, gt, stride, x_shifts, y_shifts, eps=1e-8):
        l1_target[:, 0] = gt[:, 0] / stride - x_shifts
        l1_target[:, 1] = gt[:, 1] / stride - y_shifts
        l1_target[:, 2] = torch.log(gt[:, 2] / stride + eps)
        l1_target[:, 3] = torch.log(gt[:, 3] / stride + eps)
        return l1_target

    @torch.no_grad()
    def get_assignments(
        self,
        batch_idx,
        num_gt,
        total_num_anchors,
        gt_bboxes_per_image,
        gt_classes,
        bboxes_preds_per_image,
        expanded_strides,
        x_shifts,
        y_shifts,
        cls_preds,
        bbox_preds,
        obj_preds,
        labels,
        imgs,
        mode="gpu",
    ):

        if mode == "cpu":
            print("------------CPU Mode for This Batch-------------")
            gt_bboxes_per_image = gt_bboxes_per_image.cpu().float()
            bboxes_preds_per_image = bboxes_preds_per_image.cpu().float()
            gt_classes = gt_classes.cpu().float()
            expanded_strides = expanded_strides.cpu().float()
            x_shifts = x_shifts.cpu()
            y_shifts = y_shifts.cpu()

        fg_mask, is_in_boxes_and_center = self.get_in_boxes_info(
            gt_bboxes_per_image,
            expanded_strides,
            x_shifts,
            y_shifts,
            total_num_anchors,
            num_gt,
        )

        bboxes_preds_per_image = bboxes_preds_per_image[fg_mask]
        cls_preds_ = cls_preds[batch_idx][fg_mask]
        obj_preds_ = obj_preds[batch_idx][fg_mask]
        num_in_boxes_anchor = bboxes_preds_per_image.shape[0]

        if mode == "cpu":
            gt_bboxes_per_image = gt_bboxes_per_image.cpu()
            bboxes_preds_per_image = bboxes_preds_per_image.cpu()

        pair_wise_ious = bboxes_iou(gt_bboxes_per_image, bboxes_preds_per_image, False)

        gt_cls_per_image = (
            F.one_hot(gt_classes.to(torch.int64), self.num_classes)
            .float()
            .unsqueeze(1)
            .repeat(1, num_in_boxes_anchor, 1)
        )
        pair_wise_ious_loss = -torch.log(pair_wise_ious + 1e-8)

        if mode == "cpu":
            cls_preds_, obj_preds_ = cls_preds_.cpu(), obj_preds_.cpu()

        with torch.cuda.amp.autocast(enabled=False):
            cls_preds_ = (
                cls_preds_.float().unsqueeze(0).repeat(num_gt, 1, 1).sigmoid_()
                * obj_preds_.float().unsqueeze(0).repeat(num_gt, 1, 1).sigmoid_()
            )
            pair_wise_cls_loss = F.binary_cross_entropy(
                cls_preds_.sqrt_(), gt_cls_per_image, reduction="none"
            ).sum(-1)
        del cls_preds_

        cost = (
            pair_wise_cls_loss
            + 3.0 * pair_wise_ious_loss
            + 100000.0 * (~is_in_boxes_and_center)
        )

        (
            num_fg,
            gt_matched_classes,
            pred_ious_this_matching,
            matched_gt_inds,
        ) = self.dynamic_k_matching(cost, pair_wise_ious, gt_classes, num_gt, fg_mask)
        del pair_wise_cls_loss, cost, pair_wise_ious, pair_wise_ious_loss

        if mode == "cpu":
            gt_matched_classes = gt_matched_classes.cuda()
            fg_mask = fg_mask.cuda()
            pred_ious_this_matching = pred_ious_this_matching.cuda()
            matched_gt_inds = matched_gt_inds.cuda()

        return (
            gt_matched_classes,
            fg_mask,
            pred_ious_this_matching,
            matched_gt_inds,
            num_fg,
        )

    def get_in_boxes_info(
        self,
        gt_bboxes_per_image,
        expanded_strides,
        x_shifts,
        y_shifts,
        total_num_anchors,
        num_gt,
    ):
        expanded_strides_per_image = expanded_strides[0]
        x_shifts_per_image = x_shifts[0] * expanded_strides_per_image
        y_shifts_per_image = y_shifts[0] * expanded_strides_per_image
        x_centers_per_image = (
            (x_shifts_per_image + 0.5 * expanded_strides_per_image)
            .unsqueeze(0)
            .repeat(num_gt, 1)
        )  # [n_anchor] -> [n_gt, n_anchor]
        y_centers_per_image = (
            (y_shifts_per_image + 0.5 * expanded_strides_per_image)
            .unsqueeze(0)
            .repeat(num_gt, 1)
        )

        gt_bboxes_per_image_l = (
            (gt_bboxes_per_image[:, 0] - 0.5 * gt_bboxes_per_image[:, 2])
            .unsqueeze(1)
            .repeat(1, total_num_anchors)
        )
        gt_bboxes_per_image_r = (
            (gt_bboxes_per_image[:, 0] + 0.5 * gt_bboxes_per_image[:, 2])
            .unsqueeze(1)
            .repeat(1, total_num_anchors)
        )
        gt_bboxes_per_image_t = (
            (gt_bboxes_per_image[:, 1] - 0.5 * gt_bboxes_per_image[:, 3])
            .unsqueeze(1)
            .repeat(1, total_num_anchors)
        )
        gt_bboxes_per_image_b = (
            (gt_bboxes_per_image[:, 1] + 0.5 * gt_bboxes_per_image[:, 3])
            .unsqueeze(1)
            .repeat(1, total_num_anchors)
        )

        b_l = x_centers_per_image - gt_bboxes_per_image_l
        b_r = gt_bboxes_per_image_r - x_centers_per_image
        b_t = y_centers_per_image - gt_bboxes_per_image_t
        b_b = gt_bboxes_per_image_b - y_centers_per_image
        bbox_deltas = torch.stack([b_l, b_t, b_r, b_b], 2)

        is_in_boxes = bbox_deltas.min(dim=-1).values > 0.0
        is_in_boxes_all = is_in_boxes.sum(dim=0) > 0
        # in fixed center

        center_radius = 2.5

        gt_bboxes_per_image_l = (gt_bboxes_per_image[:, 0]).unsqueeze(1).repeat(
            1, total_num_anchors
        ) - center_radius * expanded_strides_per_image.unsqueeze(0)
        gt_bboxes_per_image_r = (gt_bboxes_per_image[:, 0]).unsqueeze(1).repeat(
            1, total_num_anchors
        ) + center_radius * expanded_strides_per_image.unsqueeze(0)
        gt_bboxes_per_image_t = (gt_bboxes_per_image[:, 1]).unsqueeze(1).repeat(
            1, total_num_anchors
        ) - center_radius * expanded_strides_per_image.unsqueeze(0)
        gt_bboxes_per_image_b = (gt_bboxes_per_image[:, 1]).unsqueeze(1).repeat(
            1, total_num_anchors
        ) + center_radius * expanded_strides_per_image.unsqueeze(0)

        c_l = x_centers_per_image - gt_bboxes_per_image_l
        c_r = gt_bboxes_per_image_r - x_centers_per_image
        c_t = y_centers_per_image - gt_bboxes_per_image_t
        c_b = gt_bboxes_per_image_b - y_centers_per_image
        center_deltas = torch.stack([c_l, c_t, c_r, c_b], 2)
        is_in_centers = center_deltas.min(dim=-1).values > 0.0
        is_in_centers_all = is_in_centers.sum(dim=0) > 0

        # in boxes and in centers
        is_in_boxes_anchor = is_in_boxes_all | is_in_centers_all

        is_in_boxes_and_center = (
            is_in_boxes[:, is_in_boxes_anchor] & is_in_centers[:, is_in_boxes_anchor]
        )
        return is_in_boxes_anchor, is_in_boxes_and_center

    def dynamic_k_matching(self, cost, pair_wise_ious, gt_classes, num_gt, fg_mask):
        # Dynamic K
        # ---------------------------------------------------------------
        matching_matrix = torch.zeros_like(cost, dtype=torch.uint8)

        ious_in_boxes_matrix = pair_wise_ious
        n_candidate_k = min(10, ious_in_boxes_matrix.size(1))
        topk_ious, _ = torch.topk(ious_in_boxes_matrix, n_candidate_k, dim=1)
        dynamic_ks = torch.clamp(topk_ious.sum(1).int(), min=1)
        dynamic_ks = dynamic_ks.tolist()
        for gt_idx in range(num_gt):
            _, pos_idx = torch.topk(
                cost[gt_idx], k=dynamic_ks[gt_idx], largest=False
            )
            matching_matrix[gt_idx][pos_idx] = 1

        del topk_ious, dynamic_ks, pos_idx

        anchor_matching_gt = matching_matrix.sum(0)
        if (anchor_matching_gt > 1).sum() > 0:
            _, cost_argmin = torch.min(cost[:, anchor_matching_gt > 1], dim=0)
            matching_matrix[:, anchor_matching_gt > 1] *= 0
            matching_matrix[cost_argmin, anchor_matching_gt > 1] = 1
        fg_mask_inboxes = matching_matrix.sum(0) > 0
        num_fg = fg_mask_inboxes.sum().item()

        fg_mask[fg_mask.clone()] = fg_mask_inboxes

        matched_gt_inds = matching_matrix[:, fg_mask_inboxes].argmax(0)
        gt_matched_classes = gt_classes[matched_gt_inds]

        pred_ious_this_matching = (matching_matrix * pair_wise_ious).sum(0)[
            fg_mask_inboxes
        ]
        return num_fg, gt_matched_classes, pred_ious_this_matching, matched_gt_inds

**Module 9/10:** `src/yolox_x_detection_pipeline/yolo_pafpn.py` (carried verbatim; see the note above)

In [ ]:
#!/usr/bin/env python
# -*- encoding: utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

import torch
import torch.nn as nn

# standalone rewrite (build_notebook.py): `from .darknet import CSPDarknet` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .network_blocks import BaseConv, CSPLayer, DWConv` removed — names are kernel globals defined by the carried modules


class YOLOPAFPN(nn.Module):
    """
    YOLOv3 model. Darknet 53 is the default backbone of this model.
    """

    def __init__(
        self,
        depth=1.0,
        width=1.0,
        in_features=("dark3", "dark4", "dark5"),
        in_channels=[256, 512, 1024],
        depthwise=False,
        act="silu",
    ):
        super().__init__()
        self.backbone = CSPDarknet(depth, width, depthwise=depthwise, act=act)
        self.in_features = in_features
        self.in_channels = in_channels
        Conv = DWConv if depthwise else BaseConv

        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.lateral_conv0 = BaseConv(
            int(in_channels[2] * width), int(in_channels[1] * width), 1, 1, act=act
        )
        self.C3_p4 = CSPLayer(
            int(2 * in_channels[1] * width),
            int(in_channels[1] * width),
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )  # cat

        self.reduce_conv1 = BaseConv(
            int(in_channels[1] * width), int(in_channels[0] * width), 1, 1, act=act
        )
        self.C3_p3 = CSPLayer(
            int(2 * in_channels[0] * width),
            int(in_channels[0] * width),
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )

        # bottom-up conv
        self.bu_conv2 = Conv(
            int(in_channels[0] * width), int(in_channels[0] * width), 3, 2, act=act
        )
        self.C3_n3 = CSPLayer(
            int(2 * in_channels[0] * width),
            int(in_channels[1] * width),
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )

        # bottom-up conv
        self.bu_conv1 = Conv(
            int(in_channels[1] * width), int(in_channels[1] * width), 3, 2, act=act
        )
        self.C3_n4 = CSPLayer(
            int(2 * in_channels[1] * width),
            int(in_channels[2] * width),
            round(3 * depth),
            False,
            depthwise=depthwise,
            act=act,
        )

    def forward(self, input):
        """
        Args:
            inputs: input images.

        Returns:
            Tuple[Tensor]: FPN feature.
        """

        #  backbone
        out_features = self.backbone(input)
        features = [out_features[f] for f in self.in_features]
        [x2, x1, x0] = features

        fpn_out0 = self.lateral_conv0(x0)  # 1024->512/32
        f_out0 = self.upsample(fpn_out0)  # 512/16
        f_out0 = torch.cat([f_out0, x1], 1)  # 512->1024/16
        f_out0 = self.C3_p4(f_out0)  # 1024->512/16

        fpn_out1 = self.reduce_conv1(f_out0)  # 512->256/16
        f_out1 = self.upsample(fpn_out1)  # 256/8
        f_out1 = torch.cat([f_out1, x2], 1)  # 256->512/8
        pan_out2 = self.C3_p3(f_out1)  # 512->256/8

        p_out1 = self.bu_conv2(pan_out2)  # 256->256/16
        p_out1 = torch.cat([p_out1, fpn_out1], 1)  # 256->512/16
        pan_out1 = self.C3_n3(p_out1)  # 512->512/16

        p_out0 = self.bu_conv1(pan_out1)  # 512->512/32
        p_out0 = torch.cat([p_out0, fpn_out0], 1)  # 512->1024/32
        pan_out0 = self.C3_n4(p_out0)  # 1024->1024/32

        outputs = (pan_out2, pan_out1, pan_out0)
        return outputs

**Module 10/10:** `src/yolox_x_detection_pipeline/yolox.py` (carried verbatim; see the note above)

In [ ]:
#!/usr/bin/env python
# -*- encoding: utf-8 -*-
# Copyright (c) Megvii Inc. All rights reserved.

import torch.nn as nn

# standalone rewrite (build_notebook.py): `from .yolo_head import YOLOXHead` removed — names are kernel globals defined by the carried modules
# standalone rewrite (build_notebook.py): `from .yolo_pafpn import YOLOPAFPN` removed — names are kernel globals defined by the carried modules


class YOLOX(nn.Module):
    """
    YOLOX model module. The module list is defined by create_yolov3_modules function.
    The network returns loss values from three YOLO layers during training
    and detection results during test.
    """

    def __init__(self, backbone=None, head=None):
        super().__init__()
        if backbone is None:
            backbone = YOLOPAFPN()
        if head is None:
            head = YOLOXHead(80)

        self.backbone = backbone
        self.head = head

    def forward(self, x, targets=None):
        # fpn output content features of [dark3, dark4, dark5]
        fpn_outs = self.backbone(x)

        if self.training:
            assert targets is not None
            loss, iou_loss, conf_loss, cls_loss, l1_loss, num_fg = self.head(
                fpn_outs, targets, x
            )
            outputs = {
                "total_loss": loss,
                "iou_loss": iou_loss,
                "l1_loss": l1_loss,
                "conf_loss": conf_loss,
                "cls_loss": cls_loss,
                "num_fg": num_fg,
            }
        else:
            outputs = self.head(fpn_outs)

        return outputs

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the upstream GitHub release assets **at tag `0.1.1rc0`** (an immutable tag, never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `YoloxXDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "modelId": "Megvii-BaseDetection/YOLOX",
  "revision": "0.1.1rc0",
  "revisionKind": "github-release-tag",
  "sourceUrl": "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_x.pth",
  "upstreamCodeRevision": "419778480ab6ec0590e5d3831b3afb3b46ab2aa3",
  "license": "apache-2.0",
  "modelKey": "yolox-x",
  "files": [
    {
      "path": "yolox_x.pth",
      "bytes": 793463373,
      "sha256": "5652330b6ae860043f091b8f550a60c10e1129f416edfdb65c259be6caf355cf"
    }
  ],
  "totalBytes": 793463373
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = YoloxXDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. What the pretrained detector does, and where it fails

Before adapting anything, look at the model you are starting from. The carried `samples` module draws a deterministic 640×640 street scene containing five objects whose COCO labels and exact boxes it also returns: a **stop sign**, a **traffic light**, an analogue **clock**, a **sports ball** and a **bench**. Those boxes are references for a per-object `box_iou` check — they are drawn ground truth on a rendered picture, not a labelled photographic dataset, so nothing here is a mean-average-precision measurement.

Why 640×640 specifically: the pipeline letterboxes every input to that size, so an image already at it skips the resize entirely and the tensor is exactly the drawn pixels with the channels reversed. Nothing about the sample depends on which resampling filter is used.

Both thresholds are **caller-owned request parameters**, not pipeline constants. The package names two upstream pairs rather than inventing a house default: the demo pair `0.3 / 0.3` (upstream `tools/demo.py`, for looking at pictures) and the evaluation pair `0.01 / 0.65` (upstream `Exp.test_conf` / `Exp.nmsthre`, for average precision, which keeps far more low-scoring boxes because AP rewards recall). This cell uses the demo pair and passes it explicitly.

**Expect one honest failure.** The drawn football is not detected as a `sports ball` at all — the model offers `kite` and a second `clock` on that box instead. That miss is kept rather than tuned away, and the evaluation report records it as a zero.

In [ ]:
import hashlib
import io
import json

threshold = 0.3  # @param {type:"number"}
nms_threshold = 0.3  # @param {type:"number"}

scene, references = tutorial_scene()
buffer = io.BytesIO()
scene.save(buffer, format='PNG')
print({'sample_kind': 'synthetic', 'size': list(scene.size), 'sha256': hashlib.sha256(buffer.getvalue()).hexdigest()[:16],
       'references': {label: len(boxes) for label, boxes in references.items()}})

input_manifest = validate_inputs(scene, threshold=threshold, nms_threshold=nms_threshold, names=['tutorial-scene'])
print({'verdict': input_manifest['verdict'], 'findings': input_manifest['findings'], 'inputs': input_manifest['inputs']})

coco_result = pipe.detect(scene, threshold=threshold, nms_threshold=nms_threshold)
for det in coco_result['detections']:
    print(f"{det['label']:>14s} {det['score']:.3f}  [{', '.join(f'{v:.0f}' for v in det['box'])}]")

coco_report = evaluation_report(coco_result, references, sample_kind='synthetic')
print({'verdict': coco_report['verdict'], 'n_detections': coco_report['n_detections']})
for metric in coco_report['metrics']:
    print(f"  {metric['reference']:>18s}  box_iou {metric['value']:.3f}  same-label detections {metric['n_detected_same_label']}")
hits = sum(1 for m in coco_report['metrics'] if m['value'] >= 0.5)
print(f'{hits}/{len(coco_report["metrics"])} drawn objects matched at IoU >= 0.5')
scene

## 5. Two checks worth running once: channel order, and what it says about nothing

**Channel order.** Upstream reads images with `cv2.imread`, which yields **BGR**, and feeds that array to the network as raw 0–255 floats. It is an easy thing to get silently wrong, because RGB input still produces plausible-looking detections — it just produces worse ones. The cell below feeds the identical letterboxed tensor with the channels flipped so you can see the difference rather than take it on trust.

**Degenerate inputs.** A detector should be asked what it does with a blank page and with pure noise, because a model that invents confident objects on structure-free input will invent them on your input too. Run it and read the counts; this one is well-behaved at the demo threshold, and the notebook would say so either way.

In [ ]:
chw, ratio = preprocess(scene)
rgb_tensor = torch.from_numpy(chw[::-1].copy()).unsqueeze(0).to(pipe.device)
with torch.no_grad():
    rgb_raw = pipe.model(rgb_tensor)
rgb_kept = postprocess(rgb_raw, len(LABELS), conf_thre=threshold, nms_thre=nms_threshold)[0]
rgb_detections = sorted(
    ([LABELS[int(row[6])], float(row[4] * row[5])] for row in (rgb_kept.tolist() if rgb_kept is not None else [])),
    key=lambda pair: -pair[1],
)
print('BGR (upstream order):', [(d['label'], round(d['score'], 3)) for d in coco_result['detections']])
print('RGB (the mistake)   :', [(label, round(score, 3)) for label, score in rgb_detections])

degenerate = {}
for name, image in (('blank', blank_scene()), ('noise', noise_scene(0))):
    demo = pipe.detect(image, threshold=threshold, nms_threshold=nms_threshold)['detections']
    lenient = pipe.detect(image, threshold=EVAL_DETECTION_THRESHOLD, nms_threshold=EVAL_NMS_THRESHOLD)['detections']
    degenerate[name] = {'at_demo_thresholds': len(demo), 'at_evaluation_thresholds': len(lenient),
                        'top': [(d['label'], round(d['score'], 3)) for d in lenient[:3]]}
print(json.dumps(degenerate, indent=2))

## 6. Sample data for adaptation, and the validation stage

The detector above knows 80 COCO classes. Suppose you need three classes it does not have. `sign_dataset` draws a deterministic 40-image labelled set over `SIGN_CLASSES` — `stop-sign`, `yield-sign`, `speed-limit-sign` — with 1–3 signs per image at jittered positions, sizes and background tints, and returns exact boxes because it knows where it drew them.

**Keep the two vocabularies apart.** COCO's `stop sign` (with a space) is a class the *pretrained* model predicts; `stop-sign` (hyphenated) is a class in *your* vocabulary that only exists after adaptation. They are not the same label and the notebook never treats them as interchangeable. `yield-sign` and `speed-limit-sign` have no COCO counterpart at all.

`validate_dataset` is the validation stage for this path and applies exactly the ceilings `finetune` applies, so a dataset it accepts cannot be refused later: record shape, box geometry inside the image, labels drawn from the vocabulary, at most 50 boxes per image (the head's label tensor width), at most 200 images, at most 20 epochs. It returns a dataset manifest, and it reports a class with no boxes as a **finding rather than an error** — that dataset is trainable, but that class's head outputs will stay untrained and you should know before you spend the compute.

This is drawn data. A model fine-tuned on it learns these renderings; that is what a bounded tutorial adaptation is for, and it is why none of the numbers later is a claim about photographs of real signs.

In [ ]:
N_IMAGES = 40  # @param {type:"integer"}
DATASET_SEED = 0  # @param {type:"integer"}
EPOCHS = 6  # @param {type:"integer"}

records = sign_dataset(N_IMAGES, seed=DATASET_SEED)
dataset_manifest = validate_dataset(records, SIGN_CLASSES, epochs=EPOCHS)
print(json.dumps({k: v for k, v in dataset_manifest.items() if k != 'schema'}, indent=2))
print('\nrecord shape expected by finetune:', dataset_manifest['schema']['record'])
print('COCO classes the pretrained model knows :', [c for c in LABELS if 'sign' in c or c == 'clock'])
print('adaptation vocabulary (not COCO)        :', list(SIGN_CLASSES))

preview = Image.new('RGB', (480, 320))
for index, record in enumerate(records[:6]):
    preview.paste(record['image'].resize((160, 160)), (160 * (index % 3), 160 * (index // 3)))
print('\nfirst six images, and the labels of the first:', records[0]['labels'])
preview

## 7. Split, re-head, and measure the baseline *before* adapting

`split_records` is a seeded permutation into a training part and a held-out part. The split is yours, not the model's: it happens before any weight is touched and the held-out records are never shown to `finetune`, so the score in Section 9 is a score on data the adapted model has not seen.

`from_pretrained(class_names=...)` builds the same YOLOX-X and loads the same verified checkpoint, but rebuilds the classification branch for your vocabulary. It prints exactly which tensors it could not transfer — the three `head.cls_preds` weight/bias pairs, one per feature level — and everything else, including the whole backbone and the box and objectness heads, comes from the COCO checkpoint. The random initialisation is seeded, because those layers are the only untrained weights in the model and they are precisely what the baseline measures.

**The baseline is not zero, and that is the interesting part.** The classification head is random, but the box and objectness heads are COCO-trained and already know how to localise a sign-shaped thing, so the model scores some average precision by accident. Measuring it is what lets you say later that the fine-tune did something, rather than that a detector produced boxes.

In [ ]:
HOLDOUT = 0.25  # @param {type:"number"}
SEED = 0  # @param {type:"integer"}

train_records, held_out = split_records(records, holdout=HOLDOUT, seed=SEED)
print({'train': len(train_records), 'held_out': len(held_out),
       'train_boxes': sum(len(r['boxes']) for r in train_records),
       'held_out_boxes': sum(len(r['boxes']) for r in held_out)})

adapter = YoloxXDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=SIGN_CLASSES, seed=SEED)
print({'class_names': list(adapter.class_names), 'device': adapter.device, 'adapted': adapter.adapted})
print('tensors that could not transfer from the COCO checkpoint:')
for name in adapter.reinitialised:
    print('   ', name)

baseline = adapter.evaluate(held_out)
print(json.dumps({'ap': round(baseline['ap'], 4), 'ap50': round(baseline['ap50'], 4),
                  'per_class_ap50': {k: round(v, 4) for k, v in baseline['per_class_ap50'].items()},
                  'n_references': baseline['n_references'], 'max_detections': baseline['max_detections']}, indent=2))

## 8. The bounded SimOTA fine-tune

This is the cell that makes the notebook an `E2E` tutorial rather than an inference demo, and it runs in the default path.

The loss and the label assignment are **upstream's**, in the carried `yolo_head` module: put the head in training mode, hand it images and a `(class, cx, cy, w, h)` target tensor, and it runs SimOTA — building an IoU-and-classification cost between every ground-truth object and every candidate anchor point, picking a dynamic number of positives per object, and returning the IoU, objectness and classification terms already weighted. What this repository owns is the bounded loop around it: the target conversion, batching, the optimiser, the seed and the ceilings.

Three defaults are worth understanding because they were chosen by measurement, not taste:

- **The backbone is frozen.** Only the head trains (11.8 M of 99.0 M parameters). It is several times faster per step, and the repository measured that unfreezing it at this learning rate *collapses* the model to AP 0.0 — a handful of gradient steps on 30 small images is enough to destroy COCO-pretrained features. The frozen backbone is also kept in eval mode so its BatchNorm running statistics are not quietly rewritten by tutorial batches.
- **Six epochs at learning rate 1e-3.** A grid over epochs, learning rate and freezing put this at held-out AP50 1.0 against 0.355 at three epochs — measured on the sibling YOLOX-S row rather than here, and verified on this one only at the chosen configuration.
- **The L1 box term stays off.** Upstream enables it only for the last 15 of 300 epochs; switching it on for a six-epoch run would change the loss scale for no benefit. You will see `l1_loss` report 0.0 throughout, and that is correct.

Watch `total_loss` fall and `num_fg` — the number of anchor points SimOTA assigned as positives per image — stay stable. A `num_fg` collapsing toward zero would mean the assignment had stopped finding anything to match.

In [ ]:
LEARNING_RATE = 1e-3  # @param {type:"number"}
BATCH_SIZE = 2  # @param {type:"integer"}
FREEZE_BACKBONE = True  # @param {type:"boolean"}

run = adapter.finetune(
    train_records,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    freeze_backbone=FREEZE_BACKBONE,
    progress=lambda row: print(
        f"epoch {row['epoch']}/{EPOCHS}  steps {row['steps']}  total {row['total_loss']:.4f}  "
        f"iou {row['iou_loss']:.4f}  conf {row['conf_loss']:.4f}  cls {row['cls_loss']:.4f}  "
        f"l1 {row['l1_loss']:.4f}  num_fg {row['num_fg']:.2f}"
    ),
)
print(json.dumps({'optimizer': run['optimizer'], 'loss': run['loss'], 'use_l1': run['use_l1'],
                  'freeze_backbone': run['freeze_backbone'],
                  'trainable_parameters': run['trainable_parameters'],
                  'total_parameters': run['total_parameters'],
                  'epochs': run['epochs'], 'batch_size': run['batch_size'],
                  'learning_rate': run['learning_rate'], 'seed': run['seed']}, indent=2))
first, last = run['history'][0]['total_loss'], run['history'][-1]['total_loss']
print(f'total_loss {first:.4f} -> {last:.4f} over {run["epochs"]} epochs')

## 9. Evaluate on the held-out split

The same `evaluate` call as the baseline, on the same held-out records, with the same thresholds — so the two numbers are comparable and the only thing that changed is the weights.

`ap50` is average precision at IoU 0.50; `ap` is COCO's primary metric, AP@[.50:.95], the mean over ten IoU thresholds from 0.50 to 0.95 — it is always lower, because it demands progressively tighter boxes. Evaluation uses the *evaluation* thresholds (`0.01 / 0.65`) rather than the demo pair, since average precision rewards recall, and caps detections at 100 per image as COCO does.

What this number is: evidence that a bounded fine-tune on 30 drawn images moved a held-out score. What it is not: these are not a detection benchmark. The held-out split is ten images from the same generator with the same three shapes, so AP50 reaching 1.0 says the task is easy and the adaptation worked — not that the model would find a real sign in a real photograph. The AP@[.50:.95] figure, which is well below 1.0, is the more informative one: the boxes are right but not perfectly tight.

In [ ]:
adapted = adapter.evaluate(held_out)
print(json.dumps({'ap': round(adapted['ap'], 4), 'ap50': round(adapted['ap50'], 4), 'ap75': round(adapted['ap75'], 4),
                  'per_class_ap50': {k: round(v, 4) for k, v in adapted['per_class_ap50'].items()},
                  'threshold': adapted['threshold'], 'nms_threshold': adapted['nms_threshold'],
                  'max_detections': adapted['max_detections'],
                  'n_images': adapted['n_images'], 'n_references': adapted['n_references']}, indent=2))
print()
print(f"{'metric':<10s} {'baseline':>10s} {'adapted':>10s} {'change':>10s}")
for key in ('ap', 'ap50'):
    before_value, after_value = baseline[key], adapted[key]
    print(f'{key:<10s} {before_value:>10.4f} {after_value:>10.4f} {after_value - before_value:>+10.4f}')
print('\nimplementation note:', adapted['implementation'])

## 10. Inference on new data

Held-out images were drawn from the same seed as the training set. These come from a different seed entirely, so their layouts, sizes, tints and sign choices were never part of the split at all — the closest a synthetic tutorial gets to new data.

The detections use the **demo thresholds** again, because this is the "what would I actually deploy" view rather than the scoring view. Each detection is checked against the drawn truth with `box_iou` on the same label.

In [ ]:
NEW_DATA_SEED = 99  # @param {type:"integer"}

new_records = sign_dataset(3, seed=NEW_DATA_SEED)
new_data_rows = []
for index, record in enumerate(new_records):
    out = adapter.detect(record['image'], threshold=threshold, nms_threshold=0.45)
    ious = []
    for box, label in zip(record['boxes'], record['labels'], strict=True):
        same_label = [d for d in out['detections'] if d['label'] == label]
        ious.append(round(max((box_iou(d['box'], box) for d in same_label), default=0.0), 3))
    row = {'image': index, 'truth': record['labels'],
           'detections': [(d['label'], round(d['score'], 3)) for d in out['detections']],
           'same_label_iou': ious}
    new_data_rows.append(row)
    print(json.dumps(row))

contact = Image.new('RGB', (640, 214))
for index, record in enumerate(new_records):
    contact.paste(record['image'].resize((213, 213)), (213 * index, 0))
contact

## 11. Export the artifact, then reload it as if from a cold start

`save_artifact` writes one file: the adapted `state_dict` plus the metadata needed to rebuild the model — the format tag, the pinned model identity, the upstream code revision, the class vocabulary, the architecture constants and the digest of the base weights it started from. It is a plain `torch.save` of tensors and simple values, with no archive to extract and nothing that requires `weights_only=False` to read back, so there is no unpacking step to make safe.

`load_artifact` is the fresh-reload check: it rebuilds the model from the file alone, without reference to the `adapter` object still in memory, and refuses an artifact whose format tag or pinned identity does not match this package. The cell then re-scores the held-out split with the reloaded model and asserts the numbers are identical — a reload that quietly lost the adaptation would show up here as a different AP, and the assertion would stop the notebook.

The last part of the cell tampers with a copy of the artifact and confirms it is refused, because "it loads" is only evidence if something that should not load is also tried.

In [ ]:
import os
from pathlib import Path

OUTPUTS = Path('outputs')
os.makedirs('outputs', exist_ok=True)
artifact_path = OUTPUTS / ARTIFACT_FILE
descriptor = adapter.save_artifact(artifact_path, notes='standalone tutorial run')
print(json.dumps({k: v for k, v in descriptor.items() if k != 'base_state_digest'}, indent=2))

reloaded = YoloxXDetectionPipeline.load_artifact(artifact_path)
print({'source': reloaded.source, 'class_names': list(reloaded.class_names), 'adapted': reloaded.adapted})
reloaded_metrics = reloaded.evaluate(held_out)
print({'reloaded_ap': round(reloaded_metrics['ap'], 4), 'reloaded_ap50': round(reloaded_metrics['ap50'], 4)})
assert abs(reloaded_metrics['ap'] - adapted['ap']) < 1e-9, 'the reloaded artifact does not reproduce the adapted score'
assert abs(reloaded_metrics['ap50'] - adapted['ap50']) < 1e-9
print('fresh reload reproduces the adapted scores exactly')

tampered = OUTPUTS / 'tampered-artifact.pt'
payload = torch.load(artifact_path, map_location='cpu', weights_only=True)
payload['format'] = 'not-a-dimer-artifact/9'
torch.save(payload, tampered)
try:
    YoloxXDetectionPipeline.load_artifact(tampered)
except ValueError as exc:
    print('tampered artifact refused ->', exc)
else:
    raise AssertionError('a tampered artifact was accepted')
finally:
    tampered.unlink(missing_ok=True)

## 12. Machine-readable outputs and provenance

Everything the notebook established, written to `outputs/` as JSON beside the artifact: the pinned identity and manifest digest, the runtime, the input and dataset manifests, the COCO detections and their per-object IoU, the baseline and adapted metrics with the exact request that produced them, the new-data rows, and the artifact descriptor. A later reader can tell what was measured, on what, with which weights, without rerunning anything.

In [ ]:
import platform

export = {
    'notebook': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE,
              'upstream_code_revision': UPSTREAM_CODE_REVISION, 'release_asset': RELEASE_ASSET_URL,
              'manifest_sha256': [entry['sha256'] for entry in MANIFEST['files']],
              'depth': MODEL_DEPTH, 'width': MODEL_WIDTH, 'input_size': list(INPUT_SIZE)},
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__,
                'torchvision': torchvision.__version__, 'device': adapter.device,
                'cuda': torch.cuda.is_available(), 'precision': 'float32'},
    'coco_demonstration': {'input_manifest': input_manifest, 'detections': coco_result['detections'],
                           'evaluation': coco_report},
    'degenerate_inputs': degenerate,
    'channel_order': {'bgr': [(d['label'], round(d['score'], 4)) for d in coco_result['detections']],
                      'rgb': [(label, round(score, 4)) for label, score in rgb_detections]},
    'adaptation': {'dataset': dataset_manifest, 'split': {'train': len(train_records), 'held_out': len(held_out)},
                   'run': run, 'baseline': baseline, 'adapted': adapted,
                   'new_data': new_data_rows, 'artifact': descriptor},
}
path = OUTPUTS / 'yolox_x_detection_results.json'
with open(path, 'w', encoding='utf-8') as handle:
    json.dump(export, handle, indent=2, default=str)
print({'written': str(path), 'bytes': path.stat().st_size, 'artifact': str(artifact_path)})
print(sorted(p.name for p in OUTPUTS.iterdir()))

## 13. Optional: your own image, or your own labelled dataset

Both branches are off by default so the cell above completes a full `Run all` without stopping for an upload.

**`USE_BYOD_IMAGE`** runs one uploaded image through the same `validate_inputs` → `detect` → `evaluation_report` contract as the sample, with the pretrained COCO model. There are no reference boxes for your image, so the evaluation report will say `not-measurable` — which is the correct answer, not a failure.

**`USE_BYOD_DATASET`** is the one that matters for an adaptation profile. Supply your own labelled records and they go through the *same* local stages the sample did: validate, split, baseline, fine-tune, evaluate. It does not degrade into inference-only just because the data is yours (NOTEBOOK_SPEC 2.0 DAT14). You supply `BYOD_CLASS_NAMES` and a list of records shaped exactly like `sign_dataset`'s output; the cell prints the dataset manifest and then repeats Sections 7–9 on your data. Labelling images is outside this notebook's scope — export boxes in xyxy pixel coordinates from whatever tool you use.

The cell rejects one deliberately incompatible input so you can see what a refusal looks like before you trust an acceptance.

In [ ]:
USE_BYOD_IMAGE = False  # @param {type:"boolean"}
USE_BYOD_DATASET = False  # @param {type:"boolean"}
BYOD_CLASS_NAMES = ['my-class-a', 'my-class-b']  # @param

# What a refusal looks like, always run: the validator names the first violated ceiling.
for description, thunk in (
    ('an image that is not a PIL image', lambda: validate_inputs('/path/to/image.png')),
    ('a box outside its image', lambda: validate_dataset(
        [{'image': blank_scene(), 'boxes': [[0, 0, 9999, 10]], 'labels': [SIGN_CLASSES[0]]}], SIGN_CLASSES)),
):
    try:
        thunk()
    except (TypeError, ValueError) as exc:
        print(f'refused {description} -> {type(exc).__name__}: {exc}')

if USE_BYOD_IMAGE:
    from google.colab import files  # type: ignore[import-not-found]

    uploaded = files.upload()
    name, data = next(iter(uploaded.items()))
    byod_image = Image.open(io.BytesIO(data))
    print(validate_inputs(byod_image, threshold=threshold, nms_threshold=nms_threshold, names=[name])['verdict'])
    byod_result = pipe.detect(byod_image, threshold=threshold, nms_threshold=nms_threshold)
    for det in byod_result['detections'][:20]:
        print(f"{det['label']:>14s} {det['score']:.3f}")
    print(evaluation_report(byod_result, None, sample_kind='byod')['verdict'])
else:
    print('BYOD image branch is off; set USE_BYOD_IMAGE = True and re-run this cell to use your own image.')

if USE_BYOD_DATASET:
    # Replace byod_records with your own labelled records:
    #   [{'image': PIL.Image, 'boxes': [[x0, y0, x1, y1], ...], 'labels': ['my-class-a', ...]}, ...]
    byod_records = []  # noqa: F841 -- supply your own
    byod_manifest = validate_dataset(byod_records, BYOD_CLASS_NAMES, epochs=EPOCHS)
    print(json.dumps({k: v for k, v in byod_manifest.items() if k != 'schema'}, indent=2))
    byod_train, byod_held = split_records(byod_records, holdout=HOLDOUT, seed=SEED)
    byod_pipe = YoloxXDetectionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, class_names=BYOD_CLASS_NAMES, seed=SEED)
    print('baseline:', {k: round(v, 4) for k, v in byod_pipe.evaluate(byod_held).items() if k in ('ap', 'ap50')})
    byod_pipe.finetune(byod_train, epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, seed=SEED)
    print('adapted :', {k: round(v, 4) for k, v in byod_pipe.evaluate(byod_held).items() if k in ('ap', 'ap50')})
    byod_pipe.save_artifact(OUTPUTS / 'byod-adapter.pt', notes='BYOD tutorial run')
else:
    print('BYOD dataset branch is off; set USE_BYOD_DATASET = True, supply byod_records and BYOD_CLASS_NAMES, and re-run this cell.')

## Interpretation and limits

**What this notebook established, in this runtime.** The pinned `yolox_x.pth` release asset was downloaded, digest-verified against a committed SHA-256 before anything unpickled it, and loaded into YOLOX-X built from vendored upstream code. The pretrained detector found four of the five drawn COCO objects with per-object IoU around 0.91–0.96 and missed the fifth entirely. A three-class vocabulary that does not exist in COCO was adapted onto it by a bounded SimOTA fine-tune of the head alone, scored against a held-out split with COCO-style average precision before and after, run on images from an unseen seed, exported as a single artifact and reloaded from that artifact to identical scores.

**What it did not establish.** Nothing here is a benchmark and nothing here is about photographs. Both datasets are drawn in code with Pillow: flat colours, hard edges, no lighting, no occlusion, no motion blur, no scale variety beyond the jitter the generator applies. A held-out AP50 of 1.0 on ten images from the same generator says the task is easy and the plumbing works; it is not evidence about real signs, and the much lower AP@[.50:.95] is the more honest summary — the boxes are right, not tight. Upstream's published 51.5 AP for YOLOX-X on COCO test-dev is neither reproduced nor checked here, and the average-precision helper carried in Section 2 is a small faithful implementation without pycocotools' area ranges, crowd handling or official matching, so its numbers are not comparable to published COCO results.

**The misses are the useful part.** The drawn football is never proposed as a `sports ball`; the model offers `kite` and a second `clock` on that box instead, and the evaluation report records a zero rather than quietly dropping the reference. Earlier probing found something sharper: a plain white disc with two hands is not read as a clock at all, and drawing it that way made the *ball* become the clock — the numerals in the sample's clock face are there because of that. Treat a detector's confidence on rendered graphics as a statement about rendered graphics.

**If you adapt this to your own data.** The three defaults that were chosen by measurement will not automatically transfer. The frozen backbone is right for a few dozen images and a few epochs; with real data and real compute, unfreeze it and lower the learning rate — the repository measured a full fine-tune at 1e-3 collapsing to AP 0.0, which is what a too-large step on pretrained features looks like. Six epochs is a tutorial budget, not a recipe. And the thresholds are yours: the demo pair for looking at results, the evaluation pair for scoring, neither calibrated for anything.

**What a green run proves.** Successful execution proves that the recorded repository revision, the pinned dependency set and the pinned checkpoint together reproduce these stages in a fresh runtime, without the repository being cloned or installed and without any DIMER worker or service. It does **not** establish benchmark superiority, fitness for any deployment, or that the adapted model generalises beyond the drawn data it was fitted to.

**Reproducibility.** Every random choice is seeded — dataset generation, the split, the re-headed classification layers and the training shuffle — and the seeds are form parameters at the top of their cells. Precision is float32 on CPU and GPU alike; no autocast, no quantisation, no compiled kernels. Re-running this notebook unchanged in an equivalent runtime should reproduce the numbers; changing `DATASET_SEED` or `SEED` will change them, which is a useful thing to do once to see how much of the result is the seed.

## References

- Ge, Z., Liu, S., Wang, F., Li, Z. and Sun, J. (2021). *YOLOX: Exceeding YOLO Series in 2021.* [arXiv:2107.08430](https://arxiv.org/abs/2107.08430) — the anchor-free design, the decoupled head and SimOTA.
- [Megvii-BaseDetection/YOLOX](https://github.com/Megvii-BaseDetection/YOLOX) — the upstream repository, Apache-2.0. The model code carried in Section 2 is vendored from it at a pinned commit; the repository's `docs/UPSTREAM.md` records the revision, the per-file digests and the three edits applied.
- [Release `0.1.1rc0`](https://github.com/Megvii-BaseDetection/YOLOX/releases/tag/0.1.1rc0) — the immutable tag whose assets carry every published YOLOX checkpoint, including the `yolox_x.pth` this notebook verifies.
- Lin, T.-Y. et al. (2014). *Microsoft COCO: Common Objects in Context.* [arXiv:1405.0312](https://arxiv.org/abs/1405.0312) — the 80 classes the pretrained checkpoint predicts, and the average-precision protocol the evaluation helper approximates.
- Repository model card: https://github.com/kurtvalcorza/yolox-x-detection-pipeline/blob/main/MODEL_CARD.md
- [`kurtvalcorza/yolox-x-detection-pipeline`](https://github.com/kurtvalcorza/yolox-x-detection-pipeline) — this notebook's source repository: the package carried in Section 2, its tests, `MODEL_CARD.md`, and `docs/release-verification.md` with the measured fine-tuning grid behind the defaults used here.